# HW6 — Expert Agent Skills for Accepting a Recovered 3D Print
### One expert procedure written twice (Markdown-only and script-assisted), run by OpenCode on the same Project 2 task

**Author:** Akshara N.S.

**Project context:** Project 2, functional recovery from FDM print failures (Bambu Lab P1S). Our loop pauses a
print when the camera sees a failure, and a person can approve a correction of the remaining layers.

**Code:** https://github.com/akshara-ns/fdm-agent-skills (skills, script, scenario, this notebook).

---

## 1. Task definition

**Purpose.** The question being analysed here is: *after our recovery loop has corrected a part mid-print, can
the part be used: KEEP, TEST_BEFORE_USE or REPRINT?* Our loop changes geometry and leaves a pause seam, so this
is the decision a person faces at the end of every recovered print, and it is the decision my HW5 interviews
said only the designer can make with tolerances in hand.

**Bounded task.** One part, one decision: the team's `bracket` test part (spec copied unchanged from our
repository), printed in PLA, paused after layer 17 and corrected. The scenario (`case/print_report.md`) gives
caliper readings, the load case (100 N at one bolt, the other clamped, required safety factor 1.5), the print
settings (15 % infill), the 6-minute pause, and a 6 mm open gap at the seam. **The print event and the
measurements are a constructed scenario, not readings from a real print;** the report says so too.

**Two versions of the same expert skill**, with the same scope, inputs and decision rules:

| | `fdm-part-acceptance-md` | `fdm-part-acceptance-scripts` |
|---|---|---|
| Contents | `SKILL.md` only | `SKILL.md`, `scripts/check_part.py`, the HW5 tool code in `scripts/fdm_core/`, `references/` |
| Calculations | by hand, every formula written out | by the script; the agent chooses the inputs and interprets |
| Reuses HW5 | the same equations and material values, as text | `check_deviated_part` and `check_cantilever` unchanged, plus range limits, a skins-only section and the seam |
| Agent permissions | shell **denied** (no code or calculators, as the assignment defines this version) | shell allowed |

**Fair comparison.** Same agent (OpenCode 2.3.0), same model (`anthropic/claude-sonnet-5-5`), same scenario files,
and the same prompt except for the skill's name. Each run starts in a fresh workspace with a fresh OpenCode data
directory, so it is a new session that can see only its own skill.

## 2. Environment

**A CPU runtime is enough.** Add the Colab secret `ANTHROPIC_API_KEY` (key icon on the left, notebook access
on). The two agent runs together cost well under a dollar; the exact cost is printed after each run. Everything
except §5 and §6 runs without the key.

In [ ]:
# OpenCode (pinned) and the MCP Python SDK + my HW5 server for the independent check in section 7
import shutil
if not shutil.which("npm"):                      # Colab normally ships Node; install it if this runtime does not
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!npm install -g --silent @opencode/cli@2.3.0 2>&1 | tail -2
%pip install -q "mcp>=2.3,<3" "fdm-mcp @ git+https://github.com/akshara-ns/fdm-mcp-server"

In [ ]:
# Fail fast: confirm the tools this notebook needs are available in THIS runtime.
import importlib.util, json, os, pathlib, re, shutil, subprocess, sys, time
OPENCODE = shutil.which("opencode")
assert OPENCODE, "opencode not found: re-run the install cell."
print(subprocess.run([OPENCODE, "--version"], capture_output=True, text=True).stdout.strip())
missing = [p for p in ("mcp", "fdm_mcp") if importlib.util.find_spec(p) is None]
assert not missing, f"Missing packages: {missing}. Re-run the install cell."
print("Python", sys.version.split()[0], "| MCP SDK and fdm_mcp available.")

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception as e:
    print("No ANTHROPIC_API_KEY secret found:", type(e).__name__, "- sections 5 and 6 will not run.")
print("Anthropic key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

ROOT = pathlib.Path("/content/hw6")
for d in ["skills/fdm-part-acceptance-md", "skills/fdm-part-acceptance-scripts/scripts/fdm_core",
          "skills/fdm-part-acceptance-scripts/references", "case", "check"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
%cd /content/hw6

## 3. The two skills

### 3.1 Markdown-only skill: `fdm-part-acceptance-md/SKILL.md`

The whole procedure is in one document: when to use it and when to hand over, the inputs with units and valid
ranges and what to do when each is missing, how to normalise limits and pick the deciding reading, the cantilever
and seam equations with data-sheet values, the decision rules in order, verification steps, and when a person
must review. Its worked example uses a different part (a shelf hook), so it shows the method without the answer.

In [ ]:
%%writefile skills/fdm-part-acceptance-md/SKILL.md
---
name: fdm-part-acceptance-md
description: Decide whether an FDM 3D-printed part that was dented, corrected, paused or resumed mid-print can still be used - KEEP, TEST_BEFORE_USE or REPRINT - from caliper measurements, the part's function spec, its load case and the repair seam. Markdown-only version - work everything out by hand from this document; do not run code, scripts or calculators.
---

# FDM part acceptance after an in-print correction (Markdown-only)

You are acting as a mechanical engineer who signs off printed parts. The part did not come out as designed:
our recovery loop (or a person) paused it, dented it, corrected the remaining layers or resumed it. Your job is
to decide, with reasons a person can check, whether it can be used.

**This version has no tools.** Do every calculation by hand in your answer, showing each formula with numbers
and units substituted, so that a reader can redo it. Do not run Python, a shell, a calculator or an MCP tool.

## 1. When to use this skill, and when not to

Use it when a finished (or nearly finished) part deviates from its design and someone asks "can we use it?",
and the part's function can be judged from a few dimensions plus, at most, one cantilever-like load case.

Do **not** decide, and say so, when any of these hold; hand the case to a person instead:
- The part is safety-critical (holds a person, hangs over people, holds pressure, carries hot liquid), or the
  load is impact, vibration or many cycles (fatigue). Static calculations do not cover these.
- The part works near or above its heat deflection temperature (PLA 57 °C, PETG 69 °C, ABS-M30 104 °C).
- The load path is not a simple bending arm, a clamp or a fit (for example a thin shell in compression, a
  snap-fit, a thread). Say the strength is not assessed.

## 2. Inputs you need (with units and valid ranges)

| Input | Unit | Valid range | If missing |
|---|---|---|---|
| Material (PLA, PETG or ABS) | - | these three only | Ask. Do not assume another material's strength. |
| Each functional feature: measured value(s) | mm | 0-500 | Ask. **Never estimate a measurement.** |
| Each functional feature: limit from the designer (nominal ± tolerance, a range, or a one-sided min/max) | mm | tolerance 0.01-10 | Ask. Only the designer knows it. For a purely cosmetic feature you may use ±1.0 mm and say so. |
| Load (force or mass hung) | N or kg | 0.01-5000 N | Ask. Without a load the strength cannot be judged (then the decision is at best TEST_BEFORE_USE). |
| Where the load acts and where the part is held | mm | - | Use the centre-to-centre distance if the clamp edge is unknown (conservative), and say so. |
| Arm width b and thickness t (as printed) | mm | t 0.8-100, b 1-200 | Use the thinnest reading near the support. |
| Print orientation relative to the bending stress | flat / upright | - | Ask, or infer from "printed flat on the bed" and state the inference. |
| Infill and shells | %, mm | - | If infill < 100 % and walls do not fill the section, use the skins-only section (Step 4). |
| Required safety factor | - | 1-10 | Use the spec's value; if none, use 2.0 and say so. |
| Repair seam height, pause length, visible gaps | mm, min | - | If a pause or correction happened, a seam exists even if nobody saw it. |
| Reprint time and filament | h, g | - | Optional; used only to inform the person. |

Convert before you start: 1 kg hung = 9.81 N; 1 in = 25.4 mm; 1 MPa = 1 N/mm². Keep everything in N, mm, MPa.

When you cannot ask (a non-interactive run), continue with the most conservative assumption, label it
**ASSUMED** in the answer, and add a human check for it. Never let an assumption make the result look better.

## 3. Step-by-step procedure

### Step 0 - Hard stops (no calculation needed)
- The part detached, is spaghetti, or a feature is missing entirely: **REPRINT**.
- A visible crack or open gap runs through the most-stressed region of a load-bearing feature: **REPRINT**.
- Material or load unknown and cannot be asked: you cannot reach KEEP.

### Step 1 - Put every limit into one form
- *Range* `[lo, hi]` (for example a hole that must be 4.0-6.5 mm): nominal = (lo + hi)/2, tolerance =
  (hi - lo)/2. Example: [6.0, 6.4] becomes 6.2 ± 0.2 mm.
- *One-sided* (a minimum wall, a maximum height): check the reading directly against the bound; margin =
  reading - min (or max - reading).
- *Hole position*: a position tolerance `tol_center` is the allowed distance of each hole centre from its true
  position. If you can only measure the centre-to-centre spacing, you cannot tell which hole moved, so be
  conservative: the spacing error must be no more than `tol_center` (as if all of it were at one hole).
- *Cosmetic* features (outer shell, appearance): ± 1.0 mm unless the designer says otherwise. A cosmetic
  feature out of tolerance gives a caution, never a REPRINT by itself, **unless the part's function is its
  appearance** (a display or cosplay part). Ask if unclear.

### Step 2 - Choose the reading that decides
- Repeated readings: the reading **farthest from the nominal** (or closest to a one-sided bound) decides.
  For thickness in a bending arm, use the **thinnest** reading near the support.
- A dented hole becomes oval: the smallest diameter decides whether the bolt passes. Calipers read small holes
  low, so a hole near its lower limit should be checked with the actual bolt or a pin gauge.
- Measurement uncertainty: assume ± 0.05 mm for digital calipers on printed plastic. A margin smaller than
  that is **borderline**, which means TEST_BEFORE_USE (test-fit), not KEEP.

### Step 3 - Judge each feature
Deviation = used reading - nominal. Margin = tolerance - |deviation|. Status: OK if margin >= 0.05 mm,
BORDERLINE if 0 <= margin < 0.05 mm, FAIL if margin < 0. Put every feature in a table with spec, readings,
used value, deviation, margin and status.

### Step 4 - Strength of the load-bearing arm
Model the arm as a cantilever: fixed at the support (the clamp edge), point load F at the free end, length L
from the load to the clamp edge, width b, thickness t in the direction the load bends it.

- Bending moment at the support: **M = F · L** (N·mm).
- **Solid section** (100 % infill or walls fill it): I = b t³ / 12, stress **σ = M (t/2) / I = 6 F L / (b t²)**.
- **Skins-only section** (sparse infill): only the top and bottom shells of thickness s carry bending:
  I = b (t³ - (t - 2s)³) / 12, σ = M (t/2) / I. This ignores walls and infill, so it is a **lower bound** on
  strength; the solid result is the upper bound. Decide on the lower bound when it changes the decision.
- Strength to compare with (Bambu Lab PLA/PETG Basic TDS v2.0, Stratasys ABS-M30 data sheet):

  | Material | In-plane strength (along the lines) | E | HDT (0.45 MPa) |
  |---|---|---|---|
  | PLA | 39 MPa | 2.68 GPa | 57 °C |
  | PETG | 48 MPa | 1.46 GPa | 69 °C |
  | ABS (ABS-M30) | 30.8 MPa | 2.40 GPa | 104 °C |

  If the part was printed **flat** (layers parallel to the bending stress), use the in-plane strength. If it
  was printed **upright** (the bending stress pulls layers apart), use in-plane × 0.5, a conservative
  interlayer factor (Ahn et al. 2002 measured 10-73 % of moulded strength across bonds).
- Safety factor **SF = strength / σ**. Compare with the required factor (spec, or 2.0).
- Tip deflection **δ = F L³ / (3 E I)** with E in MPa (2.68 GPa = 2680 MPa). Flag δ > 2 % of L as a stiffness
  caution; above 10 % of L the small-deflection formulas are not valid.
- Validity: if L/t < 4 the arm is too stubby for beam theory; say the result is rough.
- Also compute SF at the **design** thickness, to show how much the deviation cost. SF scales with t² for a
  solid section, so SF_measured = SF_design × (t_measured / t_design)². Use this as a cross-check.

### Step 5 - The repair seam
Every pause, resume or correction leaves a **seam**: the plane where new plastic was laid on a layer that had
cooled. Its bond is weaker than a normal layer and gets worse the longer the pause (PLA cools below its glass
transition within a minute or two, even on a warm bed). Treat the seam's strength as **unknown, between a
normal layer and zero**.

- A seam that is a full layer crosses every section of the arm, so it is **in the load path**.
- **Flat print:** bending stress runs parallel to the seam, so the seam carries interlaminar **shear**.
  Solid section at height z above the bottom: τ = 6 F z (t - z) / (b t³) (maximum 1.5 F / (b t) at mid-height).
  If the seam lies in the sparse core between skins: τ ≈ F / (b (t - s)). A rough allowable shear is
  0.5 × the interlayer design strength (PLA: 0.5 × 0.5 × 39 = 9.75 MPa).
- **Worst case, seam fully debonded:** the two layers (thicknesses z and t - z) bend as separate beams:
  I_split = b (z³ + (t - z)³) / 12, σ = M · (max(z, t - z) / 2) / I_split, SF = in-plane strength / σ.
- **Upright print:** the seam is pulled apart directly by the bending stress σ_seam = M |z - t/2| / I. A
  calculation cannot vouch for it.
- **Visible gap** (a fingernail catches, light shows through): the bond is not intact. Moment grows linearly
  from the load point to the support, so a gap within the third of the arm nearest the load is in the
  low-moment region; elsewhere it is in the high-moment region.

### Step 6 - Decide (apply the rules in this order; list every rule that fired)
**REPRINT** if any of:
1. a fit or load-bearing feature is out of tolerance (FAIL), or a one-sided bound is violated;
2. the solid-section SF < 1;
3. an open gap at the seam lies in the high-moment region (more than L/3 from the load point);
4. an open gap at a seam that carries bending tension (upright print);
5. there is a visible gap and the fully debonded SF < 1.

**TEST_BEFORE_USE** if no REPRINT rule fired and any of:
1. a repair seam crosses the load path (always, for a full-layer seam in a loaded arm);
2. any visible seam gap in the low-moment region, or of unknown location;
3. the lower-bound SF (skins-only, or solid if solid) is below the required SF;
4. the fully debonded SF is below the required SF;
5. a feature is BORDERLINE;
6. a load-bearing feature exists but no load case was given.

**KEEP** only if no rule above fired: every feature within tolerance with margin, the lower-bound SF meets the
target, and there is no seam or defect in the load path.

Then give the reprint cost if you have the numbers: cost = filament g / 1000 × $23/kg + hours × $0.50/h
(defaults: Bambu PLA price, machine time). If a reprint is cheaper and quicker than the proof test, say so: the
decision label stays as the rules give it, but the person may reasonably choose to reprint.

### Step 7 - Verify before you answer
- Units: every stress in MPa (N/mm²), every moment in N·mm, every length in mm.
- Recompute σ by a second route (σ = M / Z with Z = b t² / 6 for a solid section) and check it matches.
- Ratio check: SF_measured / SF_design should equal (t_measured / t_design)² for a solid section.
- Plausibility: printed brackets usually have SF between 1 and 20; σ above the material strength means a
  mistake or a broken part; δ of several millimetres on a short stiff arm means a units slip.
- Re-read the spec for items you did not check (for example a volume or mass spec you cannot measure) and
  list them as **not checked**, with what measurement would check them.

### Step 8 - Recognise when a person must review
Always hand over to a person, with a clear list, when: the decision is TEST_BEFORE_USE or REPRINT; any input
was ASSUMED; there is any visible seam defect; the case is outside Section 1's scope; or your two calculation
routes disagree by more than 2 %. The person must test-fit fits with the real bolt, inspect the seam, run the
proof test and confirm tolerances and load with the designer.

**Proof test for TEST_BEFORE_USE:** hang the service load on the part for 10 minutes (or load a sacrificial
copy printed with the same pause to SF_target × load), then re-inspect the seam. Any crack, creak or growth of
a gap means REPRINT.

## 4. Worked example (a different part, to show the form)

A PLA shelf hook printed flat: arm L = 40 mm, b = 20 mm, design t = 5.0 mm, readings 4.55 and 4.50 mm, solid
infill, 2 kg hung (F = 2 × 9.81 = 19.62 N), required SF 2, a seam from a 3-minute pause at z = 2.0 mm, no gap.
- M = 19.62 × 40 = 784.8 N·mm. Use t = 4.50 mm. σ = 6 × 19.62 × 40 / (20 × 4.50²) = 4708.8 / 405 = 11.63 MPa.
- SF = 39 / 11.63 = 3.35 ≥ 2. Design: σ = 4708.8 / 500 = 9.42 MPa, SF = 4.14; check 4.14 × (4.5/5)² = 3.35 ✓.
- Seam shear: τ = 6 × 19.62 × 2.0 × 2.5 / (20 × 4.5³) = 588.6 / 1822.5 = 0.32 MPa, far below 9.75 MPa.
- Debonded: I_split = 20 × (2.0³ + 2.5³) / 12 = 20 × 23.625 / 12 = 39.4 mm⁴; σ = 784.8 × 1.25 / 39.4 = 24.9 MPa;
  SF = 39 / 24.9 = 1.57 < 2.
- Rules fired: TEST 1 (seam in load path), TEST 4 (debonded SF 1.57 < 2). Decision **TEST_BEFORE_USE**.

## 5. Answer format
1. **Decision** in one line, then the rules that fired.
2. Feature table (spec, readings, used value, deviation, margin, status).
3. Strength: every formula with numbers, solid and lower-bound SF, design SF, deflection, the cross-check.
4. Seam: location, what it carries, bonded and debonded results, gap region.
5. Assumptions (label each ASSUMED or GIVEN), spec items not checked.
6. Human checks and what result would change the decision.

### 3.2 Script-assisted skill: `fdm-part-acceptance-scripts/`

`SKILL.md` keeps the scope, the decision rules and the judgement calls (which reading, which arm length, which
section), and hands every number to `scripts/check_part.py`. It also says when to run the script, how to read its
output, and what to do when it fails (exit code 2 = fix or ask for the input, never invent it; an import error =
self-test, then the hand fallback in `references/method.md`, labelled NOT SCRIPT-VERIFIED).

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/SKILL.md
---
name: fdm-part-acceptance-scripts
description: Decide whether an FDM 3D-printed part that was dented, corrected, paused or resumed mid-print can still be used - KEEP, TEST_BEFORE_USE or REPRINT - from caliper measurements, the part's function spec, its load case and the repair seam. Script-assisted version - encode the case as JSON and run scripts/check_part.py (reuses the HW5 fdm-mcp tools check_deviated_part and check_cantilever) instead of calculating by hand.
---

# FDM part acceptance after an in-print correction (script-assisted)

You are acting as a mechanical engineer who signs off printed parts. The part did not come out as designed:
our recovery loop (or a person) paused it, dented it, corrected the remaining layers or resumed it. Decide,
with reasons a person can check, whether it can be used.

**Division of labour.** Your job is the engineering judgement: reading the spec and the report, choosing the
right inputs (which reading, which length, which section), handling what is missing, and explaining the result.
The script's job is every number and the decision rules. **Do not compute stresses or safety factors by hand
when the script can run**; copy numbers from its output. The method behind it is in `references/method.md`.

## 1. When to use this skill, and when not to
Use it when a finished part deviates from its design and someone asks "can we use it?", and its function can be
judged from a few dimensions plus at most one cantilever-like load case.

Do **not** decide, and hand the case to a person, when the part is safety-critical (holds a person, hangs over
people, holds pressure or hot liquid), the load is impact, vibration or fatigue, it works near its heat
deflection temperature (PLA 57 °C, PETG 69 °C, ABS 104 °C), or the load path is not a bending arm, a clamp or a
fit. The script cannot see any of these; you must.

## 2. Procedure

### Step 1 - Read the case and collect inputs
Read the part spec and print report completely. You need, in mm / N / kg:
- material (PLA, PETG or ABS);
- every functional feature's readings and the designer's limit;
- the load, where it acts and where the part is held; arm width and thickness readings;
- orientation (flat = bending stress along the layers; upright = bending pulls layers apart);
- infill/shells (sparse infill -> `"section": "skins"` with `skin_mm` = top/bottom shell thickness);
- the required safety factor; the seam height, pause length and any visible gap and where it is;
- reprint time and filament, if known.

**Missing information.** Never invent a measurement, tolerance or load. If one is missing, ask. If you cannot ask
(a non-interactive run), leave it out: the script refuses a fit feature without a limit and a beam without a
load (exit code 2) rather than guess. For non-critical inputs (target safety factor, cosmetic tolerance) the
script applies a stated default (2.0, ± 1.0 mm); report each default as ASSUMED.

### Step 2 - Encode the case as JSON (your judgement goes here)
Write `case.json` following `references/case_schema.md`; `references/example_case.json` shows every field for a
different part (do not copy its numbers). Expert choices the script cannot make for you:
- **Limits:** a range `[lo, hi]` goes in `range_mm`; a minimum wall in `min_mm`. A hole **position tolerance**
  `tol_center`, when you can only measure the centre-to-centre spacing, becomes a spacing feature with
  `nominal_mm` = design spacing and `tolerance_mm` = `tol_center` (conservative: as if one hole moved).
- **Readings:** pass all readings in `readings_mm`; the script uses the worst one. Use the **smallest** diameter
  of an oval (dented) hole and thickness readings taken **near the support**, where the moment is largest.
- **Arm length** `length_mm`: from the load line to the **edge of the clamp**, not to the hole centre. If the
  clamp edge is unknown, use the centre-to-centre distance (conservative) and say so.
- **Width** `width_mm`: the measured width of the section at the clamp edge (subtract a hole only if it cuts
  that section).
- **Seam:** any pause, resume or correction creates one at the resume height `z_mm`. Give
  `visible_gap.distance_from_load_mm` (distance from the load point along the arm) if a gap was seen.

### Step 3 - Run the script
```bash
python3 <skill-dir>/scripts/check_part.py case.json --out result.json
```
`<skill-dir>` is the directory this SKILL.md is in. Only the Python standard library is used (Python 3.10+).
Run `python3 <skill-dir>/scripts/check_part.py --selftest` first if you have not run it in this session: it
checks the code against hand-calculated numbers (σ = 9.42 MPa, SF = 4.14 for the HW5 demo bracket).

### Step 4 - If execution fails
- **Exit code 2, `INPUT ERROR: ...`:** the message names the field. Fix units or a typo and rerun; if the value is
  genuinely missing, ask for it (or, non-interactively, report that the decision cannot go beyond
  TEST_BEFORE_USE without it). Never fill the gap with an invented number just to make the script run.
- **`python3` not found or an import/traceback error:** check you are calling the script by its full path and
  that `scripts/fdm_core/` is next to it; rerun `--selftest`. If it still fails, follow the hand method in
  `references/method.md`, label every number **NOT SCRIPT-VERIFIED**, and ask for a person to recheck.
- **Output that contradicts the inputs** (for example a thickness in the report that is not one you gave):
  stop, fix `case.json`, rerun. Do not edit the script to get a different answer.

### Step 5 - Interpret the output
The report starts with `DECISION:` and lists every rule that fired (`REPRINT because` / `TEST because`).
- **Features:** `used` is the deciding reading; `margin` < 0 is out of tolerance; 0 to 0.05 mm is borderline.
- **Strength:** the solid section is the upper bound and the skins section the lower bound; the decision uses
  the lower bound. `design thickness ... (HW5 tool)` shows what the deviation cost.
- **Seam:** in a flat print the seam carries shear (`if bonded like a layer` is a rough margin); `if fully
  debonded` is the worst case if the seam let go. A large gap between the two is why a test is needed.
- **HW5 check_deviated_part:** the HW5 tool's own decision on the two-sided features and the solid beam. If it
  is *less* strict than the final decision, that is expected (it knows nothing about skins, seams or gaps). If it
  is *stricter*, investigate before answering.

Decision rules the script applies (identical to the Markdown-only skill), in order:
- **REPRINT** if: a fit or load-bearing feature is out of tolerance or a one-sided bound is violated; the solid
  SF < 1; an open seam gap lies in the high-moment region (more than L/3 from the load point); an open gap at a
  seam that carries bending tension (upright print); or a visible gap and the fully debonded SF < 1.
- **TEST_BEFORE_USE** if no REPRINT rule fired and: a repair seam crosses the load path; any visible seam gap in
  the low-moment region or of unknown location; the lower-bound SF is below target; the fully debonded SF is
  below target; a feature is borderline; or a load-bearing feature has no load case.
- **KEEP** only if nothing above fired.

### Step 6 - Verify before answering
- Spot-check one number by hand: σ_solid = 6 F L / (b t²) with the script's F, L, b, t. It must match to 1 %.
- Check that every value in the output's echo came from the report, and that the deciding readings are the ones
  you meant (smallest hole diameter, thinnest thickness near the clamp).
- List spec items the script did not check (for example a volume spec or a minimum wall you had no reading for)
  as **not checked**.
- If the reprint is cheaper and faster than the proof test, say so: the label stays, but the person may choose
  to reprint.

### Step 7 - Recognise when a person must review
Always hand over to a person, with the script's `Human checks` list plus anything you assumed, when the decision
is TEST_BEFORE_USE or REPRINT, any input was assumed, there is any visible seam defect, the case is outside
Section 1, or your hand spot-check disagrees with the script. The script trusts its inputs; it cannot see the
part, test-fit a bolt or proof-load it.

## 3. Equivalent MCP tool
The same HW5 core is served as the MCP tool `check_deviated_part` by the `fdm-mcp` server
(github.com/akshara-ns/fdm-mcp-server). If that server is connected you may call it as an independent check of
the two-sided features and the solid-section safety factor, but it has no range or one-sided limits, skins
section or seam check, so the script is the primary tool for this skill.

## 4. Answer format
1. **Decision** in one line, then the rules that fired (from the script).
2. The exact command you ran and the script's report (or its key lines).
3. Feature table and strength/seam numbers, copied from the output with units.
4. Your input choices (length, width, readings, section) and why; assumptions labelled ASSUMED or GIVEN; spec
   items not checked.
5. Your hand spot-check.
6. Human checks and what result would change the decision.

**Supporting script.** `check_part.py` reuses my HW5 tool code unchanged (`fdm_core/`, copied from the
`fdm-mcp` server): `check_deviated_part` judges the two-sided tolerances and re-runs `check_cantilever` at the
as-printed thickness. The script adds what this task needs and HW5 lacked: range and one-sided limits, repeated
readings and measurement uncertainty, a skins-only lower bound for sparse infill, and the repair seam (shear at
the seam and a fully debonded bound).

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/scripts/check_part.py
#!/usr/bin/env python3
"""Acceptance check for an FDM part that was dented, corrected or resumed mid-print.

Decides KEEP / TEST_BEFORE_USE / REPRINT from a case file (JSON) of caliper measurements, the part's function
spec, its load case and the repair seam. It reuses the HW5 tools unchanged (fdm_core/new_tools.check_deviated_part
for the tolerance check, fdm_core/print_tools.check_cantilever for the solid-section strength) and adds what those
tools do not cover: range and one-sided limits, repeated readings, measurement uncertainty, a skins-only section
for sparse infill, and the repair seam (shear across it and a fully debonded bound).

    python3 check_part.py case.json                 human-readable report on stdout
    python3 check_part.py case.json --out res.json  also save the full record as JSON
    python3 check_part.py --template                print an example case file
    python3 check_part.py --selftest                check the code against hand-calculated numbers

Exit codes: 0 = decision made, 2 = invalid or missing input (message says what to fix), 1 = internal error.
Only the Python standard library is needed (Python 3.10+).
"""
from __future__ import annotations

import argparse
import json
import math
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent))
from fdm_core.new_tools import check_deviated_part                       # noqa: E402  (HW5 tool 6)
from fdm_core.print_tools import MATERIALS, INTERLAYER_FACTOR, ToolInputError, G  # noqa: E402

VERSION = "1.0 (HW6)"
DEFAULT_UNCERTAINTY_MM = 0.05     # digital calipers on printed plastic: resolution 0.01, repeatability ~0.02-0.05
DEFAULT_TARGET_SF = 2.0           # same default as HW5 check_cantilever when the spec gives none
SHEAR_TO_TENSILE = 0.5            # rough shear strength / tensile strength for a printed interface (assumption)

TEMPLATE = {
    "part": "shelf hook (example only: replace every value with your own case)",
    "material": "PLA",
    "measurement_uncertainty_mm": 0.05,
    "features": [
        {"name": "screw hole diameter", "role": "fit", "readings_mm": [5.22, 5.25], "nominal_mm": 5.3,
         "tolerance_mm": 0.15},
        {"name": "rail slot width", "role": "fit", "readings_mm": [6.18], "range_mm": [6.0, 6.4]},
        {"name": "back plate wall", "role": "load_bearing", "readings_mm": [2.9], "min_mm": 2.4},
        {"name": "outer shell length", "role": "cosmetic", "readings_mm": [40.6], "nominal_mm": 40.0},
    ],
    "beam": {"length_mm": 40, "width_mm": 20, "nominal_thickness_mm": 5.0,
             "measured_thickness_readings_mm": [4.55, 4.5], "load_kg": 2, "orientation": "flat",
             "target_safety_factor": 2, "section": "solid"},
    "seam": {"z_mm": 2.0, "pause_min": 3},
    "reprint": {"time_h": 2.5, "filament_g": 38},
}


class CaseError(ValueError):
    pass


def _num(d, key, lo, hi, unit="", required=True, default=None):
    v = d.get(key, default)
    if v is None:
        if required:
            raise CaseError(f"{key} is required ({unit}).".replace(" ()", ""))
        return None
    try:
        v = float(v)
    except (TypeError, ValueError):
        raise CaseError(f"{key} must be a number in {unit or 'the stated unit'} (got {v!r}).")
    if math.isnan(v) or not (lo <= v <= hi):
        raise CaseError(f"{key} = {v:g} {unit} is outside the valid range {lo:g}-{hi:g} {unit}.")
    return v


def _readings(f, name):
    if "readings_mm" in f:
        r = f["readings_mm"]
        if not isinstance(r, list) or not r:
            raise CaseError(f"{name}: readings_mm must be a non-empty list of numbers in mm.")
        return [_num({"x": x}, "x", 0, 500, "mm") for x in r]
    if "measured_mm" in f:
        return [_num(f, "measured_mm", 0, 500, "mm")]
    raise CaseError(f"{name}: give the measurement as measured_mm or readings_mm (mm). Never estimate it.")


# --------------------------------------------------------------------------------------------------------------
# 1. Features: normalise every spec form, pick the worst reading, then let the HW5 tool judge the two-sided ones
# --------------------------------------------------------------------------------------------------------------
def normalise_features(features, u):
    two_sided, one_sided, rows = [], [], []
    for i, f in enumerate(features, 1):
        name = str(f.get("name") or f"feature {i}")
        role = str(f.get("role") or "").lower()
        if role not in ("cosmetic", "fit", "load_bearing"):
            raise CaseError(f"{name}: role must be cosmetic, fit or load_bearing (got {f.get('role')!r}).")
        reads = _readings(f, name)
        if "range_mm" in f:
            lo, hi = (float(x) for x in f["range_mm"])
            if not lo < hi:
                raise CaseError(f"{name}: range_mm must be [low, high] with low < high.")
            nom, tol, spec = (lo + hi) / 2, (hi - lo) / 2, f"{lo:g} to {hi:g} mm"
        elif "nominal_mm" in f:
            nom = _num(f, "nominal_mm", 0.1, 500, "mm")
            tol = f.get("tolerance_mm")
            if tol is None and role != "cosmetic":
                raise CaseError(f"{name}: tolerance_mm (or range_mm/min_mm/max_mm) is required for a {role} "
                                f"feature; only the designer knows it. Ask for it instead of guessing.")
            tol = 1.0 if tol is None else _num(f, "tolerance_mm", 0.01, 10, "mm")
            spec = f"{nom:g} ± {tol:g} mm"
        elif "min_mm" in f or "max_mm" in f:
            lo = _num(f, "min_mm", 0, 500, "mm", required=False)
            hi = _num(f, "max_mm", 0, 500, "mm", required=False)
            worst = min(reads) if lo is not None else max(reads)
            margin = (worst - lo) if lo is not None else (hi - worst)
            status = "FAIL" if margin < 0 else ("BORDERLINE" if margin < u else "OK")
            if status == "FAIL" and role == "cosmetic":
                status = "CAUTION"
            one_sided.append({"name": name, "role": role, "spec": (f">= {lo:g} mm" if lo is not None else
                                                                    f"<= {hi:g} mm"),
                              "readings_mm": reads, "used_mm": worst, "margin_mm": round(margin, 3),
                              "status": status})
            continue
        else:
            raise CaseError(f"{name}: give the limit as nominal_mm + tolerance_mm, range_mm [lo, hi], min_mm or max_mm.")
        worst = max(reads, key=lambda x: abs(x - nom))          # the reading farthest from the centre decides
        two_sided.append({"name": name, "role": role, "nominal_mm": round(nom, 4), "measured_mm": worst,
                          "tolerance_mm": round(tol, 4)})
        rows.append({"name": name, "spec": spec, "readings_mm": reads, "used_mm": worst,
                     "margin_mm": round(tol - abs(worst - nom), 3)})
    return two_sided, one_sided, rows


# --------------------------------------------------------------------------------------------------------------
# 2. Strength: solid section (HW5 check_cantilever, via check_deviated_part), skins-only lower bound, seam
# --------------------------------------------------------------------------------------------------------------
def beam_numbers(b, material):
    props = MATERIALS[material]
    L = _num(b, "length_mm", 5, 400, "mm")
    w = _num(b, "width_mm", 1, 200, "mm")
    t0 = _num(b, "nominal_thickness_mm", 0.8, 100, "mm")
    if "measured_thickness_readings_mm" in b:
        reads = [_num({"x": x}, "x", 0.8, 100, "mm") for x in b["measured_thickness_readings_mm"]]
    else:
        reads = [_num(b, "measured_thickness_mm", 0.8, 100, "mm")]
    t = min(reads)                                              # thinnest reading in the high-moment region
    if b.get("load_N") is None and b.get("load_kg") is None:
        raise CaseError("beam: give the service load as load_N or load_kg. Ask for it; do not assume a load.")
    F = _num(b, "load_N", 0.01, 5000, "N") if b.get("load_N") is not None else _num(b, "load_kg", 0.001, 500, "kg") * G
    orient = str(b.get("orientation") or "").lower()
    if orient not in ("flat", "upright"):
        raise CaseError("beam.orientation must be 'flat' (bending stress runs along the layers) or 'upright' "
                        "(bending stress pulls layers apart).")
    target = _num(b, "target_safety_factor", 1, 10, "", required=False) or DEFAULT_TARGET_SF
    k_z = _num(b, "interlayer_factor", 0.1, 1, "", required=False) or INTERLAYER_FACTOR
    section = str(b.get("section") or "solid").lower()
    if section not in ("solid", "skins"):
        raise CaseError("beam.section must be 'solid' (100 % infill or all walls) or 'skins' (sparse infill).")
    s = _num(b, "skin_mm", 0.2, 10, "mm", required=(section == "skins"))
    if section == "skins" and 2 * s >= t:
        section = "solid"                                       # skins meet: the section is solid anyway

    xy = props["xy_strength_MPa"]
    strength = xy if orient == "flat" else xy * k_z
    M = F * L
    I_solid = w * t ** 3 / 12
    I = I_solid if section == "solid" else w * (t ** 3 - (t - 2 * s) ** 3) / 12
    sigma = M * (t / 2) / I
    E = props["E_GPa"] * 1000
    return {"L": L, "w": w, "t0": t0, "t": t, "t_reads": reads, "F": F, "M": M, "orient": orient, "target": target,
            "k_z": k_z, "section": section, "s": s, "xy": xy, "strength": strength, "I_solid": I_solid, "I": I,
            "sigma": sigma, "sf": strength / sigma, "sf_solid": strength / (M * (t / 2) / I_solid),
            "defl": F * L ** 3 / (3 * E * I), "E": E}


def seam_numbers(seam, bn):
    t, w, F, M = bn["t"], bn["w"], bn["F"], bn["M"]
    z = _num(seam, "z_mm", 0.05, 100, "mm")
    if z >= t:
        raise CaseError(f"seam.z_mm = {z:g} mm is above the measured thickness {t:g} mm.")
    interlayer = bn["xy"] * bn["k_z"]                         # tensile design strength across layers
    shear_allow = SHEAR_TO_TENSILE * interlayer
    out = {"z_mm": z, "offset_from_neutral_axis_mm": z - t / 2}
    if bn["orient"] == "flat":
        # bending stress runs parallel to the seam; the seam carries interlaminar shear
        if bn["section"] == "skins" and bn["s"] < z < t - bn["s"]:
            tau = F / (w * (t - bn["s"]))                       # sandwich: core shear ~ V / (b d)
            where = "in the sparse core between the skins"
        else:
            tau = 6 * F * z * (t - z) / (w * t ** 3)            # solid rectangle: V Q / (I b)
            where = "in a solid region"
        out.update({"shear_at_seam_MPa": tau, "where": where, "seam_shear_allowable_MPa": shear_allow,
                    "sf_seam_shear_if_bonded": shear_allow / tau, "seam_carries": "interlaminar shear"})
        # fully debonded bound: the two layers bend as separate beams sharing the moment by stiffness
        t1, t2 = z, t - z
        I_split = w * (t1 ** 3 + t2 ** 3) / 12
        sig_split = M * (max(t1, t2) / 2) / I_split
        out.update({"sf_if_seam_fully_debonded": bn["xy"] / sig_split})
    else:
        sig_seam = M * abs(z - t / 2) / bn["I_solid"]
        out.update({"seam_carries": "bending tension across the layers",
                    "normal_stress_at_seam_MPa": sig_seam,
                    "sf_seam_if_bonded_like_other_layers": interlayer / sig_seam if sig_seam else float("inf"),
                    "sf_if_seam_fully_debonded": 0.0})
    gap = seam.get("visible_gap")
    if gap:
        d = _num(gap, "distance_from_load_mm", 0, 400, "mm", required=False)
        out["visible_gap_length_mm"] = _num(gap, "length_mm", 0.1, 400, "mm", required=False)
        if d is None:
            out["gap_region"] = "unknown"
        else:
            out["gap_moment_fraction"] = min(1.0, d / bn["L"])
            out["gap_region"] = "high_moment" if d > bn["L"] / 3 else "low_moment"
    out["pause_min"] = _num(seam, "pause_min", 0, 10000, "min", required=False)
    return out


# --------------------------------------------------------------------------------------------------------------
# 3. Decision (the same rules as SKILL.md)
# --------------------------------------------------------------------------------------------------------------
def decide(hw5, one_sided, rows, bn, sn, u):
    reprint, test, notes = [], [], []
    for r in hw5["features"]:
        if r["status"] == "FAIL":
            reprint.append(f"{r['name']} is out of tolerance ({r['deviation_mm']:+.3f} mm vs ±{r['tolerance_mm']:g}).")
    for r in rows:
        if 0 <= r["margin_mm"] < u:
            test.append(f"{r['name']} is within {u:g} mm of its limit (margin {r['margin_mm']:.3f} mm): "
                        f"check it with the real mating part.")
    for r in one_sided:
        if r["status"] == "FAIL":
            reprint.append(f"{r['name']} violates {r['spec']} (reading {r['used_mm']:g} mm).")
        elif r["status"] == "BORDERLINE":
            test.append(f"{r['name']} is within {u:g} mm of {r['spec']}.")
    if bn:
        if bn["sf_solid"] < 1:
            reprint.append(f"Even the solid section breaks: safety factor {bn['sf_solid']:.2f} < 1.")
        elif bn["sf"] < bn["target"]:
            test.append(f"Lower-bound safety factor {bn['sf']:.2f} is below the target {bn['target']:g}.")
        if bn["defl"] > 0.02 * bn["L"]:
            notes.append(f"Tip deflection {bn['defl']:.2f} mm is over 2 % of the arm length; check stiffness.")
    if sn:
        if bn and bn["orient"] == "upright":
            if sn.get("gap_region"):
                reprint.append("An open gap at a seam that carries the bending tension (upright print).")
            else:
                test.append("The seam carries the bending tension directly (upright print): proof test required.")
        if sn.get("gap_region") == "high_moment":
            reprint.append("An open gap at the seam lies in the high-moment two-thirds of the arm.")
        elif sn.get("gap_region") in ("low_moment", "unknown"):
            test.append("An open gap at the seam means the bond is not intact"
                        + (" (location unknown)." if sn["gap_region"] == "unknown" else " (low-moment end)."))
        if sn.get("sf_if_seam_fully_debonded", 99) < 1 and sn.get("gap_region"):
            reprint.append(f"With the visible gap, a fully debonded seam would break the arm "
                           f"(safety factor {sn['sf_if_seam_fully_debonded']:.2f}).")
        elif bn and sn.get("sf_if_seam_fully_debonded", 99) < bn["target"]:
            test.append(f"If the seam let go completely the safety factor would drop to "
                        f"{sn['sf_if_seam_fully_debonded']:.2f}, below the target {bn['target']:g}.")
        test.append("A pause/resume seam crosses the load path; its bond strength is unknown.")
    if bn is None and any(r["role"] == "load_bearing" for r in hw5["features"]):
        test.append("A load-bearing feature was listed but no load case was given, so strength was not checked.")
    decision = "REPRINT" if reprint else ("TEST_BEFORE_USE" if test else "KEEP")
    return decision, reprint, test, notes


def run_case(case):
    if not isinstance(case, dict):
        raise CaseError("The case file must be a JSON object.")
    material = str(case.get("material") or "").upper()
    if material not in MATERIALS:
        raise CaseError(f"material must be one of {', '.join(MATERIALS)} (got {case.get('material')!r}).")
    u = _num(case, "measurement_uncertainty_mm", 0, 1, "mm", required=False) or DEFAULT_UNCERTAINTY_MM
    feats = case.get("features") or []
    if not feats:
        raise CaseError("Give at least one measured feature.")
    two_sided, one_sided, rows = normalise_features(feats, u)

    b = case.get("beam")
    bn = beam_numbers(b, material) if b else None
    hw5_beam = None
    if bn:
        hw5_beam = {"material": material, "length_mm": bn["L"], "width_mm": bn["w"],
                    "nominal_thickness_mm": bn["t0"], "measured_thickness_mm": bn["t"], "load_N": bn["F"],
                    "orientation": bn["orient"], "target_safety_factor": bn["target"], "interlayer_factor": bn["k_z"]}
    sn = seam_numbers(case["seam"], bn) if (case.get("seam") and bn) else None
    if case.get("seam") and not bn:
        raise CaseError("A seam was given without a beam load case; give the load case so the seam can be checked.")
    rp = case.get("reprint") or {}
    hw5 = check_deviated_part(features=two_sided or [{"name": "(none)", "role": "cosmetic", "nominal_mm": 1,
                                                      "measured_mm": 1}],
                              beam=hw5_beam, repaired_in_load_path=bool(sn),
                              reprint_time_h=rp.get("time_h"), reprint_filament_g=rp.get("filament_g"))
    decision, why_reprint, why_test, notes = decide(hw5, one_sided, rows, bn, sn, u)
    return {"tool": "check_part", "version": VERSION, "decision": decision,
            "reasons_reprint": why_reprint, "reasons_test": why_test, "notes": notes,
            "features_two_sided": [{**r, **{k: v for k, v in h.items() if k in ("deviation_mm", "tolerance_used_pct",
                                                                                   "status")}}
                                   for r, h in zip(rows, hw5["features"])] if two_sided else [],
            "features_one_sided": one_sided,
            "beam": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in bn.items()} if bn else None,
            "seam": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in sn.items()} if sn else None,
            "hw5_check_deviated_part": {"decision": hw5["decision"], "verdict": hw5["verdict"],
                                        "summary": hw5["summary"], "results": hw5["results"]},
            "measurement_uncertainty_mm": u,
            "assumptions": [
                f"Material strengths from data sheets ({material}: in-plane {MATERIALS[material]['xy_strength_MPa']:g} "
                f"MPa); across layers = in-plane x interlayer_factor ({bn['k_z'] if bn else INTERLAYER_FACTOR:g}, "
                "Ahn et al. 2002 range 0.1-0.73).",
                "Cantilever with a rigid clamp and a static point load: sigma = M c / I, M = F L.",
                "The thinnest thickness reading is used; the worst reading of each feature decides it.",
                "Skins-only section ignores walls and infill (lower bound); solid section is the upper bound.",
                f"Seam shear allowable = {SHEAR_TO_TENSILE:g} x the interlayer tensile design value (rough); the "
                "real bond after a pause can be anywhere from that to zero, so the debonded bound is also shown.",
                "Range limits are converted exactly to nominal = midpoint, tolerance = half-width for the HW5 tool.",
            ],
            "human_checks": [
                "Test-fit the real bolt or mating part in every fit feature; calipers read small holes low.",
                "Inspect the seam all round (fingernail, magnifier) and photograph any gap.",
                "For TEST_BEFORE_USE: proof-load to the service load for 10 min (or a sacrificial copy to "
                "target_SF x load), then re-inspect the seam; any crack, creak or growth of a gap means REPRINT.",
                "Confirm the tolerances, load and clamp position with the designer; the script trusts its inputs.",
                "Decide with the owner whether the test is worth it when a reprint is cheap.",
            ]}


def report(r):
    out = [f"DECISION: {r['decision']}", ""]
    for x in r["reasons_reprint"]:
        out.append(f"  REPRINT because: {x}")
    for x in r["reasons_test"]:
        out.append(f"  TEST because:    {x}")
    for x in r["notes"]:
        out.append(f"  note:            {x}")
    out += ["", "Features (worst reading decides):"]
    for f in r["features_two_sided"]:
        out.append(f"  {f['name']:<34} spec {f['spec']:<16} used {f['used_mm']:>7.3f}  dev {f['deviation_mm']:+.3f}  "
                   f"margin {f['margin_mm']:+.3f} mm  {f['status']}")
    for f in r["features_one_sided"]:
        out.append(f"  {f['name']:<34} spec {f['spec']:<16} used {f['used_mm']:>7.3f}  margin {f['margin_mm']:+.3f} mm  "
                   f"{f['status']}")
    b = r["beam"]
    if b:
        out += ["", f"Strength ({b['orient']} print, {b['section']} section, F = {b['F']:.1f} N, L = {b['L']:g} mm, "
                    f"b = {b['w']:g} mm, t = {b['t']:g} mm (min of {b['t_reads']})):",
                f"  moment M = F L           = {b['M']:.0f} N mm",
                f"  stress, solid section    = {b['M'] * (b['t'] / 2) / b['I_solid']:.2f} MPa -> safety factor "
                f"{b['sf_solid']:.2f}",
                (f"  stress, skins section    = {b['sigma']:.2f} MPa -> safety factor {b['sf']:.2f} (lower bound)"
                 if b["section"] == "skins" else "  (solid section: no skins-only lower bound needed)"),
                f"  decision uses safety factor {b['sf']:.2f} vs target {b['target']:g} "
                f"(strength {b['strength']:g} MPa, {'in-plane' if b['orient'] == 'flat' else 'across layers'})",
                f"  design thickness {b['t0']:g} mm, solid  -> safety factor "
                f"{r['hw5_check_deviated_part']['results']['beam_safety_factor_design']['value']:.2f} (HW5 tool)",
                f"  tip deflection            = {b['defl']:.2f} mm ({100 * b['defl'] / b['L']:.1f} % of L)"]
    s = r["seam"]
    if s:
        out += ["", f"Repair seam at z = {s['z_mm']:g} mm ({s['offset_from_neutral_axis_mm']:+.2f} mm from the neutral "
                    f"axis), carries {s['seam_carries']}:"]
        if "shear_at_seam_MPa" in s:
            out += [f"  shear at seam            = {s['shear_at_seam_MPa']:.3f} MPa ({s['where']})",
                    f"  if bonded like a layer   -> shear safety factor {s['sf_seam_shear_if_bonded']:.1f} "
                    f"(rough allowable {s['seam_shear_allowable_MPa']:.2f} MPa)"]
        out.append(f"  if fully debonded        -> bending safety factor {s['sf_if_seam_fully_debonded']:.2f}")
        if "gap_region" in s:
            out.append(f"  visible gap: {s.get('visible_gap_length_mm', '?')} mm, region {s['gap_region']}")
    h = r["hw5_check_deviated_part"]
    out += ["", f"HW5 check_deviated_part (two-sided features + solid beam): {h['decision']}"]
    if "reprint_cost_usd" in h["results"]:
        out.append(f"Reprint: {h['results']['reprint_time_h']['value']:g} h, cost ${h['results']['reprint_cost_usd']['value']:.2f} "
                   f"(filament + machine time)")
    out += ["", "Human checks:"] + [f"  - {x}" for x in r["human_checks"]]
    return "\n".join(out)


def selftest():
    """Numbers checked by hand (HW5 report, Table 1) plus the formulas this script adds."""
    case = {"material": "PLA", "features": [{"name": "hole", "role": "fit", "measured_mm": 5.22, "nominal_mm": 5.3,
                                             "tolerance_mm": 0.15}],
            "beam": {"length_mm": 40, "width_mm": 20, "nominal_thickness_mm": 5.0, "measured_thickness_mm": 5.0,
                     "load_kg": 2, "orientation": "flat", "target_safety_factor": 2}}
    r = run_case(case)
    sigma = 6 * 2 * 9.81 * 40 / (20 * 25)                         # 9.4176 MPa
    assert abs(r["beam"]["sigma"] - sigma) < 1e-3, r["beam"]["sigma"]
    assert abs(r["beam"]["sf"] - 39 / sigma) < 1e-3                 # 4.14
    assert r["decision"] == "TEST_BEFORE_USE" or r["decision"] == "KEEP"
    # skins section: I = b (t^3 - (t-2s)^3) / 12
    case["beam"].update({"section": "skins", "skin_mm": 1.0})
    r = run_case(case)
    I = 20 * (125 - 27) / 12
    assert abs(r["beam"]["sigma"] - 2 * 9.81 * 40 * 2.5 / I) < 1e-3
    # range limits convert exactly
    two, _, rows = normalise_features([{"name": "h", "role": "fit", "readings_mm": [4.86, 5.02],
                                        "range_mm": [4.0, 6.5]}], 0.05)
    assert two[0]["nominal_mm"] == 5.25 and two[0]["tolerance_mm"] == 1.25 and two[0]["measured_mm"] == 4.86
    # a missing tolerance is refused, not guessed
    try:
        normalise_features([{"name": "h", "role": "fit", "measured_mm": 5, "nominal_mm": 5.3}], 0.05)
        raise AssertionError("missing tolerance accepted")
    except CaseError:
        pass
    print("selftest passed: stress 9.42 MPa and safety factor 4.14 match the hand calculation; skins section, "
          "range conversion and missing-tolerance refusal behave as documented.")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    ap.add_argument("case", nargs="?", help="case file (JSON)")
    ap.add_argument("--out", help="save the full result record as JSON")
    ap.add_argument("--template", action="store_true", help="print an example case file and exit")
    ap.add_argument("--selftest", action="store_true", help="check the code against hand-calculated numbers")
    a = ap.parse_args(argv)
    if a.template:
        print(json.dumps(TEMPLATE, indent=2))
        return 0
    if a.selftest:
        selftest()
        return 0
    if not a.case:
        ap.error("give a case file, or --template / --selftest")
    try:
        case = json.loads(pathlib.Path(a.case).read_text())
        r = run_case(case)
    except FileNotFoundError:
        print(f"INPUT ERROR: case file not found: {a.case}", file=sys.stderr)
        return 2
    except json.JSONDecodeError as e:
        print(f"INPUT ERROR: the case file is not valid JSON ({e}).", file=sys.stderr)
        return 2
    except (CaseError, ToolInputError) as e:
        print(f"INPUT ERROR: {e}", file=sys.stderr)
        return 2
    print(report(r))
    if a.out:
        pathlib.Path(a.out).write_text(json.dumps(r, indent=2, ensure_ascii=False))
        print(f"\nFull record saved to {a.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/scripts/fdm_core/__init__.py

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/scripts/fdm_core/print_tools.py
# Copied unchanged from HW5 (github.com/akshara-ns/fdm-mcp-server, src/fdm_mcp/) so the skill runs without the MCP server.
"""Analytical tools for planning FDM (material-extrusion) 3D prints.

Four deterministic tools, each a plain function that takes numbers with units and returns a structured record:

  estimate_print_job      material, filament length, print time, energy and cost of one print
  check_cantilever        will a printed bracket/hook hold a load? (bending, with layer orientation)
  check_container         can a printed cup hold a liquid? (pressure, wall, heat)
  compare_failure_costs   expected waste per good part: unmonitored vs detect-and-stop vs detect-and-adapt

Every record has the same shape, so the agent and the UI can treat the tools alike:
  {"tool", "verdict", "summary", "results": {key: {"label", "value", "unit"}}, "checks": [...],
   "inputs": {...}, "defaults_used": [...], "assumptions": [...], "warnings": [...]}

Out-of-range input raises ToolInputError with a message written for the user.
"""
from __future__ import annotations

import math

G = 9.81                       # m/s^2
FILAMENT_DIAMETER_MM = 1.75

# Values for printed specimens from manufacturer technical data sheets (TDS); full citations in the notebook (§3.5):
#   PLA, PETG: Bambu Lab PLA Basic / PETG Basic TDS v2.0 (ISO 527 tensile, 100 % infill, annealed specimens;
#              heat deflection temperature ISO 75 at 0.45 MPa).
#   ABS:       Stratasys ABS-M30 data sheet (ASTM D638 yield, XZ "on edge" and ZX "upright"; HDT at 66 psi = 0.45 MPa).
# "z_strength_tds_MPa" is the data sheet's across-layer value. Those coupons are solid and annealed, the best case;
# as-printed parts are weaker across layers, so the checks use xy_strength x INTERLAYER_FACTOR instead (see below).
MATERIALS = {
    "PLA":  {"density": 1.24, "E_GPa": 2.68, "xy_strength_MPa": 39.0, "z_strength_tds_MPa": 35.0,
             "hdt_C": 57.0, "price_per_kg": 23.0},
    "PETG": {"density": 1.25, "E_GPa": 1.46, "xy_strength_MPa": 48.0, "z_strength_tds_MPa": 39.0,
             "hdt_C": 69.0, "price_per_kg": 23.0},
    "ABS":  {"density": 1.05, "E_GPa": 2.40, "xy_strength_MPa": 30.8, "z_strength_tds_MPa": 27.5,
             "hdt_C": 103.8, "price_per_kg": 23.0},
}

# Design strength across layers as a share of the in-plane strength. Ahn et al. (2002) measured FDM parts failing
# along inter-road bonds at 10-73 % of the moulded strength; 0.5 is a conservative design choice for as-printed
# parts (thin walls, sparse infill, imperfect bonding), not a measured constant. It is an input of the tools.
INTERLAYER_FACTOR = 0.5


class ToolInputError(ValueError):
    """Raised for input outside a tool's valid range; the message is shown to the user."""


# ----------------------------------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------------------------------
def _material(name):
    key = str(name or "PLA").strip().upper()
    if key not in MATERIALS:
        raise ToolInputError(f"Material '{name}' is not supported. Choose one of: {', '.join(MATERIALS)}.")
    return key, MATERIALS[key]


def _check_range(name, value, lo, hi, unit=""):
    if value is None:
        raise ToolInputError(f"{name} is required.")
    try:
        value = float(value)
    except (TypeError, ValueError):
        raise ToolInputError(f"{name} must be a number (got {value!r}).")
    if not (lo <= value <= hi) or math.isnan(value):
        raise ToolInputError(f"{name} = {value:g} {unit} is outside the valid range {lo:g}–{hi:g} {unit}.".replace("  ", " "))
    return value


def _r(x, nd=2):
    return round(float(x), nd)


def _res(label, value, unit, nd=2):
    return {"label": label, "value": _r(value, nd), "unit": unit}


def _worst(statuses):
    for s in ("FAIL", "CAUTION", "OK"):
        if s in statuses:
            return s
    return "OK"


def _defaults(given: dict, defaults: dict, used: list):
    """Fill None arguments from `defaults`, recording each default that was applied."""
    out = {}
    for k, d in defaults.items():
        v = given.get(k)
        if v is None:
            out[k] = d
            used.append(f"{k} = {d}")
        else:
            out[k] = v
    return out


# ----------------------------------------------------------------------------------------------------
# Tool 1: material, time, energy and cost of a print
# ----------------------------------------------------------------------------------------------------
def estimate_print_job(material: str = "PLA", shape: str = "custom",
                       size_x_mm: float = None, size_y_mm: float = None, size_z_mm: float = None,
                       volume_cm3: float = None, surface_area_cm2: float = None,
                       infill_percent: float = None, wall_loops: int = None, line_width_mm: float = None,
                       layer_height_mm: float = None, print_speed_mm_s: float = None,
                       max_flow_mm3_s: float = None, motion_efficiency: float = None,
                       printer_power_W: float = None, filament_price_per_kg: float = None,
                       electricity_price_per_kWh: float = None) -> dict:
    """Estimate filament used, print time, energy and cost for one FDM print.

    Geometry: shape='box' (size_x/y/z = outer dimensions, mm), shape='cylinder' (size_x = diameter,
    size_z = height, mm), or shape='custom' (volume_cm3 and surface_area_cm2 from the CAD/slicer).
    The part is modelled as a solid shell (wall_loops x line_width thick, over the whole surface) filled
    with sparse infill. Print time = material volume / effective volumetric flow.
    """
    used = []
    mat, props = _material(material)
    p = _defaults(locals(), {"infill_percent": 15, "wall_loops": 2, "line_width_mm": 0.42,
                             "layer_height_mm": 0.2, "print_speed_mm_s": 150, "max_flow_mm3_s": 21,
                             "motion_efficiency": 0.6, "printer_power_W": 110,
                             "filament_price_per_kg": props["price_per_kg"],
                             "electricity_price_per_kWh": 0.17}, used)
    shape = str(shape or "custom").lower()
    if shape == "box":
        x = _check_range("size_x_mm", size_x_mm, 1, 350, "mm")
        y = _check_range("size_y_mm", size_y_mm, 1, 350, "mm")
        z = _check_range("size_z_mm", size_z_mm, 0.2, 350, "mm")
        V = x * y * z / 1000.0
        A = 2 * (x * y + y * z + x * z) / 100.0
    elif shape == "cylinder":
        d = _check_range("size_x_mm (diameter)", size_x_mm, 1, 350, "mm")
        z = _check_range("size_z_mm (height)", size_z_mm, 0.2, 350, "mm")
        V = math.pi * d ** 2 / 4 * z / 1000.0
        A = (math.pi * d * z + 2 * math.pi * d ** 2 / 4) / 100.0
    elif shape == "custom":
        V = _check_range("volume_cm3", volume_cm3, 0.05, 20000, "cm³")
        A = _check_range("surface_area_cm2", surface_area_cm2, 0.5, 20000, "cm²")
        a_min = (36 * math.pi * V ** 2) ** (1 / 3)       # a sphere has the least area for a given volume
        if A < a_min * 0.999:
            raise ToolInputError(f"A part of {V:g} cm³ needs at least {a_min:.1f} cm² of surface (a sphere); "
                                 f"{A:g} cm² is impossible. Check the units (cm² vs mm²).")
    else:
        raise ToolInputError("shape must be 'box', 'cylinder' or 'custom'.")

    infill = _check_range("infill_percent", p["infill_percent"], 0, 100, "%") / 100
    loops = int(_check_range("wall_loops", p["wall_loops"], 1, 10))
    w = _check_range("line_width_mm", p["line_width_mm"], 0.2, 1.2, "mm")
    h = _check_range("layer_height_mm", p["layer_height_mm"], 0.04, 0.6, "mm")
    if h > 0.8 * w:
        raise ToolInputError(f"layer_height_mm {h:g} is more than 80 % of line_width_mm {w:g}; the layers would not bond.")
    v = _check_range("print_speed_mm_s", p["print_speed_mm_s"], 5, 600, "mm/s")
    qmax = _check_range("max_flow_mm3_s", p["max_flow_mm3_s"], 1, 60, "mm³/s")
    eta = _check_range("motion_efficiency", p["motion_efficiency"], 0.1, 1.0)
    P = _check_range("printer_power_W", p["printer_power_W"], 10, 2000, "W")
    fp = _check_range("filament_price_per_kg", p["filament_price_per_kg"], 0, 500, "$/kg")
    ep = _check_range("electricity_price_per_kWh", p["electricity_price_per_kWh"], 0, 2, "$/kWh")

    shell_mm = loops * w
    V_shell = min(V, A * shell_mm / 10.0)                 # cm² x mm / 10 = cm³
    V_mat = V_shell + infill * (V - V_shell)
    mass_g = V_mat * props["density"]
    fil_area_cm2 = math.pi * (FILAMENT_DIAMETER_MM / 10) ** 2 / 4
    fil_len_m = V_mat / fil_area_cm2 / 100
    q_nominal = w * h * v
    q_eff = min(q_nominal, qmax) * eta
    t_h = V_mat * 1000 / q_eff / 3600
    energy_kWh = P * t_h / 1000
    cost_mat = mass_g / 1000 * fp
    cost_energy = energy_kWh * ep

    checks = [{"name": "Flow limit", "status": "CAUTION" if q_nominal > qmax else "OK",
               "detail": (f"Requested flow {q_nominal:.1f} mm³/s exceeds the hot end's {qmax:g} mm³/s; the "
                          f"printer slows down to the limit." if q_nominal > qmax else
                          f"Requested flow {q_nominal:.1f} mm³/s is within the {qmax:g} mm³/s limit.")}]
    warnings = []
    if V_shell >= V:
        warnings.append("The walls fill the whole part, so infill has no effect (the part prints solid).")
    if t_h > 24:
        warnings.append("Over a day of printing: one failure costs a lot, so monitoring matters (see compare_failure_costs).")

    return {
        "tool": "estimate_print_job",
        "verdict": _worst([c["status"] for c in checks]),
        "summary": (f"{mat} part of {V:.1f} cm³: about {mass_g:.0f} g of filament ({fil_len_m:.1f} m), "
                    f"{t_h:.2f} h of printing, {energy_kWh:.2f} kWh, about ${cost_mat + cost_energy:.2f} in material and energy."),
        "results": {
            "part_volume_cm3": _res("Part volume (solid)", V, "cm³"),
            "surface_area_cm2": _res("Surface area", A, "cm²"),
            "material_volume_cm3": _res("Plastic deposited (shell + infill)", V_mat, "cm³"),
            "solid_fraction_pct": _res("Plastic as a share of the solid part", 100 * V_mat / V, "%", 1),
            "filament_mass_g": _res("Filament mass", mass_g, "g", 1),
            "filament_length_m": _res(f"Filament length ({FILAMENT_DIAMETER_MM} mm)", fil_len_m, "m"),
            "effective_flow_mm3_s": _res("Effective volumetric flow", q_eff, "mm³/s"),
            "print_time_h": _res("Print time", t_h, "h"),
            "print_time_min": _res("Print time in minutes", t_h * 60, "min", 0),
            "energy_kWh": _res("Electrical energy", energy_kWh, "kWh", 3),
            "material_cost_usd": _res("Filament cost", cost_mat, "$"),
            "energy_cost_usd": _res("Electricity cost", cost_energy, "$", 3),
            "total_cost_usd": _res("Material + energy cost", cost_mat + cost_energy, "$"),
        },
        "checks": checks,
        "inputs": {"material": mat, "shape": shape, "infill_percent": infill * 100, "wall_loops": loops,
                   "line_width_mm": w, "layer_height_mm": h, "print_speed_mm_s": v, "max_flow_mm3_s": qmax,
                   "motion_efficiency": eta, "printer_power_W": P, "filament_price_per_kg": fp,
                   "electricity_price_per_kWh": ep},
        "defaults_used": used,
        "assumptions": [
            "Shell = wall_loops x line_width over the whole surface (top/bottom skins are taken as equally thick).",
            "Infill fills the rest of the volume at infill_percent; supports, brim and purge are not included.",
            "Print time = plastic volume / (min(line_width x layer_height x speed, max flow) x motion_efficiency); "
            "motion_efficiency lumps acceleration, travel and layer changes. Heat-up time is not included.",
            f"Filament diameter {FILAMENT_DIAMETER_MM} mm; average printer power is constant over the print.",
        ],
        "warnings": warnings,
    }


# ----------------------------------------------------------------------------------------------------
# Tool 2: printed cantilever (bracket, hook, clip) under an end load
# ----------------------------------------------------------------------------------------------------
def check_cantilever(material: str = "PLA", length_mm: float = None, width_mm: float = None,
                     thickness_mm: float = None, load_N: float = None, load_kg: float = None,
                     orientation: str = "flat", target_safety_factor: float = None,
                     interlayer_factor: float = None) -> dict:
    """Check a printed rectangular cantilever (shelf bracket arm, hook, clip) with a load at its free end.

    orientation='flat': the arm lies on the bed, so bending stress runs along the printed lines (strong).
    orientation='upright': the arm was printed standing up, so bending stress pulls layers apart (weak).
    thickness_mm is the depth in the direction the load bends the arm. Load as load_N or load_kg (mass hung).
    """
    used = []
    mat, props = _material(material)
    L = _check_range("length_mm", length_mm, 5, 400, "mm")
    b = _check_range("width_mm", width_mm, 1, 200, "mm")
    t = _check_range("thickness_mm", thickness_mm, 0.8, 100, "mm")
    if load_N is None and load_kg is None:
        raise ToolInputError("Give the load as load_N (newtons) or load_kg (mass hung on the end).")
    F = _check_range("load_N", load_N, 0.01, 5000, "N") if load_N is not None else \
        _check_range("load_kg", load_kg, 0.001, 500, "kg") * G
    orient = str(orientation or "flat").lower()
    if orient not in ("flat", "upright"):
        raise ToolInputError("orientation must be 'flat' (layers along the arm) or 'upright' (layers across the arm).")
    d = _defaults({"target_safety_factor": target_safety_factor, "interlayer_factor": interlayer_factor},
                  {"target_safety_factor": 2.0, "interlayer_factor": INTERLAYER_FACTOR}, used)
    sf_target = _check_range("target_safety_factor", d["target_safety_factor"], 1, 10)
    k_z = _check_range("interlayer_factor", d["interlayer_factor"], 0.1, 1.0)
    z_strength = props["xy_strength_MPa"] * k_z

    strength = props["xy_strength_MPa"] if orient == "flat" else z_strength
    E = props["E_GPa"] * 1000                              # MPa
    I = b * t ** 3 / 12                                    # mm^4
    M = F * L                                              # N mm
    sigma = M * (t / 2) / I                                # MPa = 6FL/(b t^2)
    sf = strength / sigma
    defl = F * L ** 3 / (3 * E * I)                        # mm
    F_fail = strength * b * t ** 2 / (6 * L)
    t_req = math.sqrt(6 * F * L * sf_target / (b * strength))
    other = z_strength if orient == "flat" else props["xy_strength_MPa"]
    sf_other = other / sigma

    checks = [
        {"name": "Strength", "status": "FAIL" if sf < 1 else ("CAUTION" if sf < sf_target else "OK"),
         "detail": f"Safety factor {sf:.2f} against the {strength:g} MPa {'in-plane' if orient == 'flat' else 'interlayer'} "
                   f"strength (target {sf_target:g})."},
        {"name": "Stiffness", "status": "CAUTION" if defl > 0.02 * L else "OK",
         "detail": f"Tip deflection {defl:.2f} mm is {100 * defl / L:.1f} % of the length (CAUTION above 2 %)."},
    ]
    warnings = []
    if L / t < 4:
        warnings.append("Length is less than 4x the thickness: simple beam theory underestimates shear; treat as rough.")
    if defl > 0.1 * L:
        warnings.append("Deflection over 10 % of the length: outside small-deflection theory; the result is unreliable.")

    return {
        "tool": "check_cantilever",
        "verdict": _worst([c["status"] for c in checks]),
        "summary": (f"{mat} arm {L:g} x {b:g} x {t:g} mm printed {orient}, {F:.1f} N at the tip: bending stress "
                    f"{sigma:.2f} MPa, safety factor {sf:.2f}, tip deflection {defl:.2f} mm. It breaks at about {F_fail:.0f} N."),
        "results": {
            "load_N": _res("End load", F, "N"),
            "bending_moment_Nmm": _res("Bending moment at the root", M, "N·mm", 1),
            "max_bending_stress_MPa": _res("Maximum bending stress (root surface)", sigma, "MPa"),
            "allowable_strength_MPa": _res(f"Strength used ({'along lines, XY' if orient == 'flat' else 'across layers, Z'})", strength, "MPa", 1),
            "safety_factor": _res("Safety factor", sf, "-"),
            "safety_factor_if_other_orientation": _res(f"Safety factor if printed {'upright' if orient == 'flat' else 'flat'}", sf_other, "-"),
            "breaking_load_N": _res("Estimated breaking load", F_fail, "N", 1),
            "breaking_load_kg": _res("Estimated breaking load as mass", F_fail / G, "kg", 2),
            "tip_deflection_mm": _res("Tip deflection", defl, "mm"),
            "thickness_for_target_mm": _res(f"Thickness needed for safety factor {sf_target:g}", t_req, "mm"),
        },
        "checks": checks,
        "inputs": {"material": mat, "length_mm": L, "width_mm": b, "thickness_mm": t, "load_N": _r(F, 3),
                   "orientation": orient, "target_safety_factor": sf_target, "interlayer_factor": k_z},
        "defaults_used": used,
        "assumptions": [
            "Euler-Bernoulli cantilever, fixed root, point load at the tip: sigma = 6FL/(b t^2), deflection = FL^3/(3EI).",
            "Solid printed section (enough walls/100 % infill); sparse infill makes the part weaker than this.",
            "In-plane strength from the material data sheet; strength across layers = in-plane x interlayer_factor.",
            "Tensile strength is used as the bending limit (conservative: printed flexural strength is higher).",
            "Static load at room temperature: no creep, fatigue or impact. The fixed root is perfectly rigid.",
        ],
        "warnings": warnings,
    }


# ----------------------------------------------------------------------------------------------------
# Tool 3: printed cup or container holding a liquid
# ----------------------------------------------------------------------------------------------------
def check_container(material: str = "PLA", inner_diameter_mm: float = None, height_mm: float = None,
                    wall_thickness_mm: float = None, liquid_temp_C: float = None,
                    liquid_density_kg_m3: float = None, line_width_mm: float = None,
                    interlayer_factor: float = None) -> dict:
    """Check a printed upright cylindrical cup/container: capacity, wall stress from the liquid, wall
    thickness for watertightness, and whether the liquid is hot enough to soften the plastic."""
    used = []
    mat, props = _material(material)
    D = _check_range("inner_diameter_mm", inner_diameter_mm, 5, 300, "mm")
    H = _check_range("height_mm", height_mm, 5, 300, "mm")
    t = _check_range("wall_thickness_mm", wall_thickness_mm, 0.3, 20, "mm")
    p = _defaults({"liquid_temp_C": liquid_temp_C, "liquid_density_kg_m3": liquid_density_kg_m3,
                   "line_width_mm": line_width_mm, "interlayer_factor": interlayer_factor},
                  {"liquid_temp_C": 20, "liquid_density_kg_m3": 1000, "line_width_mm": 0.42,
                   "interlayer_factor": INTERLAYER_FACTOR}, used)
    T = _check_range("liquid_temp_C", p["liquid_temp_C"], -20, 100, "°C")
    rho = _check_range("liquid_density_kg_m3", p["liquid_density_kg_m3"], 500, 2000, "kg/m³")
    w = _check_range("line_width_mm", p["line_width_mm"], 0.2, 1.2, "mm")
    k_z = _check_range("interlayer_factor", p["interlayer_factor"], 0.1, 1.0)

    r = D / 2
    cap_mL = math.pi * r ** 2 * H / 1000
    p_Pa = rho * G * H / 1000                              # pressure at the bottom of the wall
    p_MPa = p_Pa / 1e6
    hoop = p_MPa * r / t                                   # acts around the cup, along the printed lines
    axial = p_MPa * r / (2 * t)                            # upper bound on the stress across layers
    sf_hoop = props["xy_strength_MPa"] / hoop
    sf_axial = props["xy_strength_MPa"] * k_z / axial
    loops = t / w
    soft = props["hdt_C"]

    checks = [
        {"name": "Wall stress from the liquid", "status": "OK" if min(sf_hoop, sf_axial) >= 2 else "FAIL",
         "detail": f"Hoop stress {hoop:.4f} MPa (safety factor {sf_hoop:.0f}); the liquid pressure alone is not a risk."},
        {"name": "Watertight wall", "status": "OK" if loops >= 3 else ("CAUTION" if loops >= 2 else "FAIL"),
         "detail": f"The wall is about {loops:.1f} lines of {w:g} mm. At least 3 lines are needed to seal reliably; "
                   f"1 line leaks."},
        {"name": "Heat", "status": "FAIL" if T >= soft else ("CAUTION" if T >= soft - 10 else "OK"),
         "detail": f"Liquid at {T:g} °C vs {mat} softening point {soft:g} °C."},
    ]
    warnings = []
    if t > 0.1 * r:
        warnings.append("Wall thicker than a tenth of the radius: the thin-wall formula overestimates the stress (conservative).")
    if mat == "PLA" and T >= soft:
        warnings.append("Hot drinks soften PLA: the cup can sag or distort even though the pressure stress is tiny.")

    return {
        "tool": "check_container",
        "verdict": _worst([c["status"] for c in checks]),
        "summary": (f"{mat} cup, {D:g} mm inside x {H:g} mm, {t:g} mm wall: holds {cap_mL:.0f} mL; liquid "
                    f"pressure gives only {hoop:.4f} MPa hoop stress; heat check: {checks[2]['status']}, "
                    f"wall seal check: {checks[1]['status']}."),
        "results": {
            "capacity_mL": _res("Capacity when full", cap_mL, "mL", 0),
            "bottom_pressure_Pa": _res("Liquid pressure at the bottom", p_Pa, "Pa", 0),
            "hoop_stress_MPa": _res("Hoop stress in the wall (along lines)", hoop, "MPa", 4),
            "axial_stress_MPa": _res("Stress across layers (upper bound)", axial, "MPa", 4),
            "safety_factor_hoop": _res("Safety factor, hoop", sf_hoop, "-", 0),
            "safety_factor_layers": _res("Safety factor, across layers", sf_axial, "-", 0),
            "wall_lines": _res("Wall thickness in printed lines", loops, "lines", 1),
            "wall_for_3_lines_mm": _res("Wall thickness for 3 lines (watertight)", 3 * w, "mm", 2),
            "softening_point_C": _res(f"{mat} softening point (HDT)", soft, "°C", 0),
            "temperature_margin_C": _res("Margin below the softening point", soft - T, "°C", 0),
        },
        "checks": checks,
        "inputs": {"material": mat, "inner_diameter_mm": D, "height_mm": H, "wall_thickness_mm": t,
                   "liquid_temp_C": T, "liquid_density_kg_m3": rho, "line_width_mm": w, "interlayer_factor": k_z},
        "defaults_used": used,
        "assumptions": [
            "Thin-walled open cylinder, full to the brim; pressure = rho g H at the bottom, hoop stress = p r / t.",
            "The floor is supported by the table, so it carries no bending stress.",
            "Watertightness is judged by wall lines (3 or more), a rule of thumb; real sealing also depends on flow and temperature.",
            "Softening point is the heat deflection temperature; prolonged contact near it causes creep. Food safety is not assessed.",
        ],
        "warnings": warnings,
    }


# ----------------------------------------------------------------------------------------------------
# Tool 4: what failures cost, with and without monitoring
# ----------------------------------------------------------------------------------------------------
def compare_failure_costs(print_time_h: float = None, filament_g: float = None, failure_probability: float = None,
                          mean_failure_point: float = None, detection_delay_min: float = None,
                          recoverable_share: float = None, filament_price_per_kg: float = None,
                          machine_cost_per_h: float = None, jobs: int = None) -> dict:
    """Expected waste per GOOD part for three strategies when each attempt fails with probability p:
    A) unmonitored: a failure is found at the end, after the full time and filament are spent;
    B) detect and stop: a camera stops the print shortly after the failure; restart from scratch;
    C) detect and adapt: as B, but a share of detected failures is repaired in place and finishes as a usable part.
    Attempts repeat until one succeeds (geometric distribution: expected failed attempts = p/(1-p))."""
    used = []
    T = _check_range("print_time_h", print_time_h, 0.05, 500, "h")
    m = _check_range("filament_g", filament_g, 0.1, 10000, "g")
    p = _defaults({"failure_probability": failure_probability, "mean_failure_point": mean_failure_point,
                   "detection_delay_min": detection_delay_min, "recoverable_share": recoverable_share,
                   "filament_price_per_kg": filament_price_per_kg, "machine_cost_per_h": machine_cost_per_h,
                   "jobs": jobs},
                  {"failure_probability": 0.15, "mean_failure_point": 0.3, "detection_delay_min": 10,
                   "recoverable_share": 0.4, "filament_price_per_kg": 23, "machine_cost_per_h": 0.5,
                   "jobs": 100}, used)
    pf = _check_range("failure_probability", p["failure_probability"], 0, 0.9)
    phi = _check_range("mean_failure_point", p["mean_failure_point"], 0, 1)
    delay_h = _check_range("detection_delay_min", p["detection_delay_min"], 0, 600, "min") / 60
    rec = _check_range("recoverable_share", p["recoverable_share"], 0, 1)
    fp = _check_range("filament_price_per_kg", p["filament_price_per_kg"], 0, 500, "$/kg")
    mc = _check_range("machine_cost_per_h", p["machine_cost_per_h"], 0, 200, "$/h")
    n_jobs = int(_check_range("jobs", p["jobs"], 1, 1e6))

    # Waste in one failed attempt: time (h) and filament (g).
    detected_frac = min(1.0, phi + delay_h / T)            # share of the job done before the printer stops
    waste_A = (T, m)                                       # spaghetti keeps extruding to the end
    waste_B = (T * detected_frac, m * detected_frac)       # extrusion stops shortly after the failure

    def per_good(p_fail, waste):
        k = p_fail / (1 - p_fail)                          # expected failed attempts per good part
        return k, k * waste[0], k * waste[1]

    kA, hA, gA = per_good(pf, waste_A)
    kB, hB, gB = per_good(pf, waste_B)
    pC = pf * (1 - rec)                                    # failures that still end in a restart
    kC, hC, gC = per_good(pC, waste_B)

    def cost(h, g):
        return g / 1000 * fp + h * mc

    rows = {"A_unmonitored": (kA, hA, gA), "B_detect_and_stop": (kB, hB, gB), "C_detect_and_adapt": (kC, hC, gC)}
    results = {}
    for key, (k, h, g) in rows.items():
        results[f"{key}_failed_attempts_per_good_part"] = _res(f"{key}: failed attempts per good part", k, "-", 3)
        results[f"{key}_wasted_hours_per_good_part"] = _res(f"{key}: wasted printer hours per good part", h, "h", 3)
        results[f"{key}_wasted_filament_g_per_good_part"] = _res(f"{key}: wasted filament per good part", g, "g", 1)
        results[f"{key}_waste_cost_per_good_part_usd"] = _res(f"{key}: waste cost per good part", cost(h, g), "$", 3)
    for key, (k, h, g) in (("B_vs_A", rows["B_detect_and_stop"]), ("C_vs_A", rows["C_detect_and_adapt"])):
        results[f"saved_{key}_filament_g_per_good_part"] = _res(f"Filament saved per good part, {key.replace('_', ' ')}", gA - g, "g", 1)
        results[f"saved_{key}_hours_per_good_part"] = _res(f"Printer hours saved per good part, {key.replace('_', ' ')}", hA - h, "h", 3)
        results[f"saved_{key}_cost_per_good_part_usd"] = _res(f"Waste cost saved per good part, {key.replace('_', ' ')}", cost(hA, gA) - cost(h, g), "$", 2)
    results["hours_saved_B_vs_A_per_jobs_h"] = _res(f"Printer hours saved by B vs A over {n_jobs} good parts", (hA - hB) * n_jobs, "h", 1)
    results["filament_saved_B_vs_A_per_jobs_g"] = _res(f"Filament saved by B vs A over {n_jobs} good parts", (gA - gB) * n_jobs, "g", 0)
    results["filament_saved_C_vs_A_per_jobs_g"] = _res(f"Filament saved by C vs A over {n_jobs} good parts", (gA - gC) * n_jobs, "g", 0)
    results["hours_saved_C_vs_A_per_jobs_h"] = _res(f"Printer hours saved by C vs A over {n_jobs} good parts", (hA - hC) * n_jobs, "h", 1)

    warnings = []
    if phi + delay_h / T >= 1:
        warnings.append("The detection delay is as long as the rest of the print, so detection saves nothing here.")
    if pf >= 0.5:
        warnings.append("A failure rate of 50 % or more points to a process problem worth fixing before monitoring.")

    return {
        "tool": "compare_failure_costs",
        "verdict": "OK",
        "summary": (f"With p = {pf:g} per attempt, each good part wastes on average {gA:.1f} g and {hA:.2f} h unmonitored, "
                    f"{gB:.1f} g and {hB:.2f} h with detect-and-stop, and {gC:.1f} g and {hC:.2f} h with detect-and-adapt."),
        "results": results,
        "checks": [],
        "inputs": {"print_time_h": T, "filament_g": m, "failure_probability": pf, "mean_failure_point": phi,
                   "detection_delay_min": delay_h * 60, "recoverable_share": rec, "filament_price_per_kg": fp,
                   "machine_cost_per_h": mc, "jobs": n_jobs},
        "defaults_used": used,
        "assumptions": [
            "Attempts are independent with the same failure probability; repeat until one succeeds (geometric).",
            "Unmonitored failures are found at the end of the job and waste its full time and filament.",
            "A detected failure stops the printer delay minutes after it happens at mean_failure_point of the job.",
            "Detect-and-adapt: recoverable_share of failures finish as a usable part with no extra waste. This share "
            "is the project's hypothesis, not a measured value.",
            "Waste cost = filament + machine time; staff time is not counted.",
        ],
        "warnings": warnings,
    }


TOOLS = {f.__name__: f for f in (estimate_print_job, check_cantilever, check_container, compare_failure_costs)}

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/scripts/fdm_core/new_tools.py
# Copied unchanged from HW5 (github.com/akshara-ns/fdm-mcp-server, src/fdm_mcp/) so the skill runs without the MCP server.
"""Two new analytical tools added for HW5, chosen from our user interviews.

  plan_unattended_print   When will an overnight / unattended job fail unseen, how much does it waste before
                          anyone finds out, and which start time keeps the first layers watched?
  check_deviated_part     A part came out (or was repaired) off-design: feature by feature, against tolerances the
                          designer gives, is it still usable, and does a load-bearing arm keep its safety factor?

Both return the same record shape as the HW4 tools in print_tools.py, plus "decision" and "human_checks".
"""
from __future__ import annotations

import math

from .print_tools import (MATERIALS, ToolInputError, _check_range, _defaults, _r, _res, _worst,
                          check_cantilever)


# ----------------------------------------------------------------------------------------------------
# Tool 5: unattended / overnight print exposure
# ----------------------------------------------------------------------------------------------------
def _clock(name, s):
    """'HH:MM' (24 h) -> hours after midnight."""
    try:
        hh, mm = str(s).strip().split(":")
        hh, mm = int(hh), int(mm)
    except Exception:
        raise ToolInputError(f"{name} must be a 24-hour time 'HH:MM' (got {s!r}).")
    if not (0 <= hh <= 23 and 0 <= mm <= 59):
        raise ToolInputError(f"{name} = {s!r} is not a valid time; use 00:00–23:59.")
    return hh + mm / 60


def _fmt(h):
    """hours after day-0 midnight -> 'HH:MM (+d)'."""
    day, rem = divmod(h, 24)
    mins = int(round(rem * 60))
    if mins == 24 * 60:
        day, mins = day + 1, 0
    return f"{mins // 60:02d}:{mins % 60:02d}" + (f" (+{int(day)} d)" if day >= 1 else "")


def _exposure(start, T, m, win_a, win_len, p, early_share, early_h, detection, cover, delay_h, check_h, fp, mc,
              n=2000):
    """Expected outcomes of one attempt started at clock time `start` (hours)."""
    end = start + T

    def unattended(t):
        return ((t - win_a) % 24) < win_len

    def human_discovery(t):
        if unattended(t):
            return t + (win_len - ((t - win_a) % 24))       # found when the person is back
        return t + check_h / 2                             # found at the next look, on average

    # Failure-time density over the job: early_share inside the first early_h, the rest spread uniformly.
    e = min(early_h, T)
    pts = []
    for i in range(n):
        u = (i + 0.5) / n
        pts.append((start + u * e, early_share / n))
        pts.append((start + u * T, (1 - early_share) / n))

    w_sum = g_sum = d_sum = un_sum = blk_sum = 0.0
    worst = 0.0
    for tf, w in pts:
        t_h = human_discovery(tf)
        if detection == "auto_stop":
            t_auto = tf + delay_h
            stop = cover * min(t_auto, end) + (1 - cover) * min(t_h, end)
            found = cover * min(t_auto, t_h) + (1 - cover) * t_h
        else:
            stop = min(t_h, end)
            found = t_h
        frac = (stop - start) / T
        w_sum += w * frac
        d_sum += w * (found - tf)
        un_sum += w * (1.0 if unattended(tf) else 0.0)
        blk_sum += w * max(0.0, found - end)               # finished/stopped but nobody can restart it yet
        worst = max(worst, t_h - tf)                       # longest a failure can go unseen
    waste_frac = w_sum
    g = p * waste_frac * m
    h = p * waste_frac * T
    # share of the print that runs while nobody is around
    steps = 1000
    un_share = sum(unattended(start + (k + 0.5) / steps * T) for k in range(steps)) / steps
    first_watched = not any(unattended(start + (k + 0.5) / 50 * e) for k in range(50))
    return {"end": end, "waste_frac": waste_frac, "g": g, "h": h, "cost": g / 1000 * fp + h * mc,
            "delay_h": d_sum, "p_unattended": un_sum, "blocked_h": blk_sum, "un_share": un_share,
            "first_watched": first_watched, "worst_h": worst, "end_unattended": unattended(end)}


def plan_unattended_print(print_time_h: float = None, filament_g: float = None, start_time: str = None,
                          away_from: str = None, back_at: str = None, detection: str = "none",
                          failure_probability: float = None, early_failure_share: float = None,
                          early_window_h: float = None, detection_coverage: float = None,
                          detection_delay_min: float = None, check_interval_min: float = None,
                          filament_price_per_kg: float = None, machine_cost_per_h: float = None,
                          earliest_start: str = None, latest_start: str = None) -> dict:
    """Exposure of one print that runs partly while nobody is watching (overnight, or a lab left empty).

    The person is away from `away_from` to `back_at` every day (the window may cross midnight). A failure while
    they are present is noticed at their next look (on average half of check_interval_min); while they are away
    it is noticed when they come back. With detection='auto_stop' (camera detection that pauses the printer),
    a share `detection_coverage` of failures stops `detection_delay_min` after it happens. A failed job keeps
    extruding until it is stopped or the G-code ends. If earliest_start and latest_start are given, every
    start time between them (15-minute steps) is evaluated and the one with the lowest expected waste cost
    among those whose first layers are watched is recommended.
    """
    used = []
    T = _check_range("print_time_h", print_time_h, 0.05, 200, "h")
    m = _check_range("filament_g", filament_g, 0.1, 10000, "g")
    if start_time is None or away_from is None or back_at is None:
        raise ToolInputError("start_time, away_from and back_at are required ('HH:MM', 24-hour clock).")
    s = _clock("start_time", start_time)
    a = _clock("away_from", away_from)
    b = _clock("back_at", back_at)
    win_len = (b - a) % 24
    if win_len == 0:
        raise ToolInputError("away_from and back_at are the same time; give the period when nobody is watching.")
    det = str(detection or "none").lower()
    if det not in ("none", "auto_stop"):
        raise ToolInputError("detection must be 'none' (you find failures yourself) or 'auto_stop' "
                             "(the printer's camera detection pauses the job).")
    p = _defaults({"failure_probability": failure_probability, "early_failure_share": early_failure_share,
                   "early_window_h": early_window_h, "detection_coverage": detection_coverage,
                   "detection_delay_min": detection_delay_min, "check_interval_min": check_interval_min,
                   "filament_price_per_kg": filament_price_per_kg, "machine_cost_per_h": machine_cost_per_h},
                  {"failure_probability": 0.15, "early_failure_share": 0.5, "early_window_h": 0.5,
                   "detection_coverage": 0.6, "detection_delay_min": 10, "check_interval_min": 60,
                   "filament_price_per_kg": 23, "machine_cost_per_h": 0.5}, used)
    if det == "none":
        used[:] = [u for u in used if not u.startswith(("detection_coverage", "detection_delay_min"))]
    pf = _check_range("failure_probability", p["failure_probability"], 0, 0.9)
    es = _check_range("early_failure_share", p["early_failure_share"], 0, 1)
    eh = _check_range("early_window_h", p["early_window_h"], 0.05, 5, "h")
    cov = _check_range("detection_coverage", p["detection_coverage"], 0, 1)
    dly = _check_range("detection_delay_min", p["detection_delay_min"], 0, 600, "min") / 60
    chk = _check_range("check_interval_min", p["check_interval_min"], 1, 480, "min") / 60
    fp = _check_range("filament_price_per_kg", p["filament_price_per_kg"], 0, 500, "$/kg")
    mc = _check_range("machine_cost_per_h", p["machine_cost_per_h"], 0, 200, "$/h")

    args = (T, m, a, win_len, pf, es, eh, det, cov, dly, chk, fp, mc)
    r = _exposure(s, *args)

    best = None
    if earliest_start is not None or latest_start is not None:
        if earliest_start is None or latest_start is None:
            raise ToolInputError("Give both earliest_start and latest_start to search for a start time.")
        e0 = _clock("earliest_start", earliest_start)
        span = (_clock("latest_start", latest_start) - e0) % 24
        cands = []
        for k in range(int(span * 4) + 1):
            st = e0 + k * 0.25
            rr = _exposure(st, *args, n=400)
            cands.append((not rr["first_watched"], round(rr["cost"], 4), k, st, rr))
        cands.sort()
        _, _, _, best_s, best_r = cands[0]
        best = {"start": best_s, **best_r}

    checks = [
        {"name": "First layers watched", "status": "OK" if r["first_watched"] else "CAUTION",
         "detail": (f"The first {min(eh, T):g} h (where {es:.0%} of failures are assumed to happen) runs while you "
                    f"are present." if r["first_watched"] else
                    f"The first {min(eh, T):g} h runs while nobody is watching; most adhesion failures start there.")},
        {"name": "Unseen failure time", "status": "CAUTION" if r["delay_h"] > 2 else "OK",
         "detail": f"If it fails, on average {r['delay_h']:.1f} h pass before anyone knows (CAUTION above 2 h)."},
    ]
    if det == "none" and r["un_share"] > 0.5:
        checks.append({"name": "No automatic stop", "status": "CAUTION",
                       "detail": f"{r['un_share']:.0%} of the print runs unattended with nothing that can stop a failure."})

    results = {
        "finish_time": {"label": "Finish time (clock)", "value": _fmt(r["end"]), "unit": "HH:MM"},
        "unattended_share_pct": _res("Share of the print that runs unattended", 100 * r["un_share"], "%", 0),
        "unattended_hours": _res("Unattended printing hours", r["un_share"] * T, "h"),
        "p_failure_while_unattended_pct": _res("If it fails: chance the failure happens while nobody is there",
                                               100 * r["p_unattended"], "%", 0),
        "mean_hours_until_known": _res("If it fails: mean time until someone knows", r["delay_h"], "h"),
        "mean_idle_hours_after_stop": _res("If it fails: mean idle printer time before it can be restarted",
                                           r["blocked_h"], "h"),
        "worst_case_hours_unseen": _res("Worst case: longest a failure can go unnoticed", r["worst_h"], "h"),
        "expected_wasted_filament_g": _res("Expected wasted filament for this attempt", r["g"], "g", 1),
        "expected_wasted_hours": _res("Expected wasted printer hours for this attempt", r["h"], "h"),
        "expected_waste_cost_usd": _res("Expected waste cost for this attempt", r["cost"], "$"),
    }
    if best is not None:
        results.update({
            "recommended_start": {"label": "Recommended start time in your window", "value": _fmt(best["start"]),
                                  "unit": "HH:MM"},
            "recommended_finish": {"label": "Finish time with the recommended start", "value": _fmt(best["end"]),
                                   "unit": "HH:MM"},
            "recommended_expected_waste_cost_usd": _res("Expected waste cost with the recommended start",
                                                        best["cost"], "$"),
            "recommended_mean_hours_until_known": _res("Mean time until a failure is known, recommended start",
                                                       best["delay_h"], "h"),
        })
    warnings = []
    if r["end_unattended"]:
        warnings.append("The job ends while you are away: a finished part waits on the bed and the printer stays idle.")
    if best is not None and not best["first_watched"]:
        warnings.append("No start time in your window lets you watch the first layers.")
    warnings.append("Hardware damage (a jammed or cracked nozzle) and fire risk are not modelled; follow your "
                    "space's rules on unattended printing.")

    summary = (f"A {T:g} h job started at {_fmt(s)} finishes at {_fmt(r['end'])}; {r['un_share']:.0%} of it runs "
               f"unattended. If it fails ({pf:.0%} chance), on average {r['delay_h']:.1f} h pass before anyone knows; "
               f"expected waste for this attempt is {r['g']:.1f} g and {r['h']:.2f} printer-hours "
               f"(${r['cost']:.2f}).")
    if best is not None:
        summary += f" Best start in your window: {_fmt(best['start'])} (expected waste ${best['cost']:.2f})."

    return {
        "tool": "plan_unattended_print",
        "verdict": _worst([c["status"] for c in checks]),
        "decision": (f"start at {_fmt(best['start'])}" if best is not None else
                     ("watch the first layers before leaving" if not r["first_watched"] else "start as planned")),
        "summary": summary,
        "results": results,
        "checks": checks,
        "inputs": {"print_time_h": T, "filament_g": m, "start_time": start_time, "away_from": away_from,
                   "back_at": back_at, "detection": det, "failure_probability": pf, "early_failure_share": es,
                   "early_window_h": eh, "detection_coverage": cov if det == "auto_stop" else None,
                   "detection_delay_min": dly * 60 if det == "auto_stop" else None, "check_interval_min": chk * 60,
                   "filament_price_per_kg": fp, "machine_cost_per_h": mc,
                   "earliest_start": earliest_start, "latest_start": latest_start},
        "defaults_used": used,
        "assumptions": [
            "One attempt fails with probability failure_probability; early_failure_share of failures happen in the "
            "first early_window_h (bed adhesion), the rest uniformly over the job. Both are assumptions, not data.",
            "While present you notice a failure at your next look (half of check_interval_min on average); while away, "
            "when you come back. The same away window repeats every day.",
            "A failed job keeps extruding (spaghetti) until it is stopped or the G-code ends; the whole attempt is "
            "restarted, never resumed.",
            "Auto-stop catches detection_coverage of failures detection_delay_min after they start; the rest behave as "
            "with no detection.",
            "Waste cost = filament + machine time; staff time, energy for re-heating and hardware damage are not counted.",
        ],
        "warnings": warnings,
        "human_checks": [
            "Watch the first layer yourself (or check the camera) before leaving, whatever the tool says.",
            "Check that unattended printing is allowed in your space and that the printer has thermal-runaway protection.",
            "Replace the default failure_probability with your own printer's record if you have one.",
        ],
    }


# ----------------------------------------------------------------------------------------------------
# Tool 6: is a deviated (or repaired) part still usable?
# ----------------------------------------------------------------------------------------------------
ROLES = ("cosmetic", "fit", "load_bearing")
COSMETIC_DEFAULT_TOL_MM = 1.0


def check_deviated_part(features: list = None, beam: dict = None, repaired_in_load_path: bool = False,
                        reprint_time_h: float = None, reprint_filament_g: float = None,
                        filament_price_per_kg: float = None, machine_cost_per_h: float = None) -> dict:
    """Decide whether a part that came out different from its design can still be used.

    features: 1–30 items {name, role, nominal_mm, measured_mm, tolerance_mm}. role is 'cosmetic' (outer shells,
    walls, looks), 'fit' (holes, slots, mating faces) or 'load_bearing'. tolerance_mm is the allowed |measured −
    nominal|; it is required for fit and load_bearing features because only the designer knows it, and defaults to
    1.0 mm for cosmetic ones. beam (optional): the load-bearing arm as {material, length_mm, width_mm,
    nominal_thickness_mm, measured_thickness_mm, load_N or load_kg, orientation, target_safety_factor}; the
    cantilever check is run for the design and the as-printed thickness. Decision: KEEP, TEST_BEFORE_USE or REPRINT.
    """
    used = []
    if not features:
        raise ToolInputError("Give at least one feature: {name, role, nominal_mm, measured_mm, tolerance_mm}.")
    if not isinstance(features, list) or len(features) > 30:
        raise ToolInputError("features must be a list of 1–30 items.")

    rows, statuses = [], []
    for i, f in enumerate(features, 1):
        if not isinstance(f, dict):
            f = dict(f)
        name = str(f.get("name") or f"feature {i}")
        role = str(f.get("role") or "").lower()
        if role not in ROLES:
            raise ToolInputError(f"{name}: role must be one of {', '.join(ROLES)} (got {f.get('role')!r}).")
        nom = _check_range(f"{name}: nominal_mm", f.get("nominal_mm"), 0.1, 500, "mm")
        meas = _check_range(f"{name}: measured_mm", f.get("measured_mm"), 0.0, 500, "mm")
        tol = f.get("tolerance_mm")
        if tol is None:
            if role != "cosmetic":
                raise ToolInputError(f"{name}: tolerance_mm is required for a {role} feature. Only the designer knows "
                                     f"how much error it can take; give the allowed deviation in mm.")
            tol = COSMETIC_DEFAULT_TOL_MM
            used.append(f"{name}: tolerance_mm = {COSMETIC_DEFAULT_TOL_MM} (cosmetic default)")
        tol = _check_range(f"{name}: tolerance_mm", tol, 0.01, 10, "mm")
        dev = meas - nom
        use = abs(dev) / tol
        if use <= 1:
            st = "OK"
        else:
            st = "CAUTION" if role == "cosmetic" else "FAIL"
        statuses.append(st)
        rows.append({"name": name, "role": role, "nominal_mm": nom, "measured_mm": meas, "deviation_mm": _r(dev, 3),
                     "tolerance_mm": tol, "tolerance_used_pct": _r(100 * use, 0), "status": st})

    results, checks, warnings = {}, [], []
    n_fail = sum(r["status"] == "FAIL" for r in rows)
    n_cos = sum(r["status"] == "CAUTION" for r in rows)
    checks.append({"name": "Dimensions within designer tolerances",
                   "status": "FAIL" if n_fail else ("CAUTION" if n_cos else "OK"),
                   "detail": (f"{n_fail} fit/load-bearing feature(s) out of tolerance; " if n_fail else "") +
                             (f"{n_cos} cosmetic feature(s) beyond {COSMETIC_DEFAULT_TOL_MM:g} mm or their tolerance."
                              if n_cos else ("all features within tolerance." if not n_fail else ""))})

    has_lb = any(r["role"] == "load_bearing" for r in rows)
    sf_status = None
    if beam:
        b = dict(beam)
        t0 = _check_range("beam.nominal_thickness_mm", b.get("nominal_thickness_mm"), 0.8, 100, "mm")
        t1 = _check_range("beam.measured_thickness_mm", b.get("measured_thickness_mm"), 0.8, 100, "mm")
        common = {k: b.get(k) for k in ("material", "length_mm", "width_mm", "load_N", "load_kg", "orientation",
                                        "target_safety_factor", "interlayer_factor")}
        common["material"] = common["material"] or "PLA"
        nominal = check_cantilever(thickness_mm=t0, **common)
        actual = check_cantilever(thickness_mm=t1, **common)
        sf0 = nominal["results"]["safety_factor"]["value"]
        sf1 = actual["results"]["safety_factor"]["value"]
        target = actual["inputs"]["target_safety_factor"]
        used += [f"beam: {d}" for d in actual["defaults_used"]]
        warnings += [f"beam: {w}" for w in actual["warnings"]]
        results.update({
            "beam_safety_factor_design": _res("Arm safety factor, design thickness", sf0, "-"),
            "beam_safety_factor_as_printed": _res("Arm safety factor, as-printed thickness", sf1, "-"),
            "beam_strength_retained_pct": _res("Bending strength retained (∝ thickness²)", 100 * (t1 / t0) ** 2, "%", 0),
            "beam_breaking_load_as_printed_N": _res("Estimated breaking load, as printed",
                                                     actual["results"]["breaking_load_N"]["value"], "N", 1),
            "beam_tip_deflection_as_printed_mm": _res("Tip deflection, as printed",
                                                      actual["results"]["tip_deflection_mm"]["value"], "mm"),
        })
        sf_status = "FAIL" if sf1 < 1 else ("CAUTION" if sf1 < target else "OK")
        checks.append({"name": "Load-bearing arm, as printed", "status": sf_status,
                       "detail": f"Safety factor {sf1:.2f} as printed vs {sf0:.2f} as designed (target {target:g})."})
    elif has_lb:
        warnings.append("A load-bearing feature was listed without `beam`; its strength was not checked, only its size.")

    if repaired_in_load_path:
        checks.append({"name": "Repair in the load path", "status": "CAUTION",
                       "detail": "Layers were changed or re-started where load passes; the bond there is unknown, so "
                                 "the calculated safety factor is not enough on its own."})

    # Decision
    if n_fail or sf_status == "FAIL":
        decision = "REPRINT"
    elif (has_lb and (sf_status in (None, "CAUTION"))) or repaired_in_load_path:
        decision = "TEST_BEFORE_USE"
    else:
        decision = "KEEP"

    if reprint_time_h is not None or reprint_filament_g is not None:
        p = _defaults({"filament_price_per_kg": filament_price_per_kg, "machine_cost_per_h": machine_cost_per_h},
                      {"filament_price_per_kg": 23, "machine_cost_per_h": 0.5}, used)
        rt = _check_range("reprint_time_h", reprint_time_h, 0.05, 500, "h")
        rg = _check_range("reprint_filament_g", reprint_filament_g, 0.1, 10000, "g")
        fp = _check_range("filament_price_per_kg", p["filament_price_per_kg"], 0, 500, "$/kg")
        mc = _check_range("machine_cost_per_h", p["machine_cost_per_h"], 0, 200, "$/h")
        results["reprint_cost_usd"] = _res("Cost of a reprint (filament + machine time)", rg / 1000 * fp + rt * mc, "$")
        results["reprint_time_h"] = _res("Printer time for a reprint", rt, "h")

    results["features_checked"] = _res("Features checked", len(rows), "-", 0)
    results["features_out_of_tolerance"] = _res("Fit/load-bearing features out of tolerance", n_fail, "-", 0)
    worst = max(rows, key=lambda r: r["tolerance_used_pct"] if r["role"] != "cosmetic" else -1)
    if worst["role"] != "cosmetic":
        results["worst_functional_tolerance_used_pct"] = _res(f"Tolerance used by the tightest functional feature "
                                                              f"({worst['name']})", worst["tolerance_used_pct"], "%", 0)

    reason = {"REPRINT": "a fit or load-bearing feature is out of tolerance or the arm would break",
              "TEST_BEFORE_USE": ("layers were changed in the load path, so a calculation alone cannot vouch for "
                                  "its strength" if repaired_in_load_path else
                                  "dimensions pass but the load-bearing strength is unchecked or below target"),
              "KEEP": "every feature is within the designer's tolerance" +
                      (" and the arm keeps its target safety factor" if sf_status == "OK" else "")}[decision]
    return {
        "tool": "check_deviated_part",
        "verdict": _worst([c["status"] for c in checks]),
        "decision": decision,
        "summary": f"Decision: {decision}, because {reason}. "
                   f"{len(rows)} feature(s) checked, {n_fail} functional feature(s) out of tolerance"
                   + (f"; arm safety factor {results['beam_safety_factor_as_printed']['value']:.2f} as printed"
                      if beam else "") + ".",
        "results": results,
        "features": rows,
        "checks": checks,
        "inputs": {"features": rows and [{k: r[k] for k in ("name", "role", "nominal_mm", "measured_mm",
                                                              "tolerance_mm")} for r in rows],
                   "beam": beam, "repaired_in_load_path": bool(repaired_in_load_path),
                   "reprint_time_h": reprint_time_h, "reprint_filament_g": reprint_filament_g},
        "defaults_used": used,
        "assumptions": [
            "A feature passes when |measured − nominal| ≤ tolerance_mm; tolerances come from the designer.",
            f"Cosmetic features (shells, outer walls) default to ±{COSMETIC_DEFAULT_TOL_MM:g} mm and can only give a CAUTION.",
            "The arm check reuses check_cantilever with the measured thickness; it assumes a solid section and the same "
            "material and layer bonding as the design, which a repaired region may not have.",
            "The tool judges geometry and calculated strength only, not surface finish, porosity or material behaviour.",
        ],
        "warnings": warnings,
        "human_checks": [
            "Measure the features yourself (calipers or a test fit); the tool trusts the numbers it is given.",
            "For TEST_BEFORE_USE, proof-load the part to the target load (or a sacrificial copy) before relying on it.",
            "Confirm the tolerances with whoever designed the part.",
        ],
    }

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/references/case_schema.md
# case.json fields for scripts/check_part.py

All lengths in mm, forces in N (or `load_kg` for a hung mass), times in h or min. Unknown keys are ignored.

| Field | Type | Required | Meaning |
|---|---|---|---|
| `part` | string | no | Name, for the record. |
| `material` | `"PLA"`, `"PETG"`, `"ABS"` | yes | Selects the data-sheet strength and stiffness. |
| `measurement_uncertainty_mm` | number 0-1 | no (0.05) | Margins smaller than this are borderline. |
| `features` | list | yes, >= 1 | One item per measured feature (below). |
| `beam` | object | if anything carries load | The load-bearing arm as a cantilever (below). |
| `seam` | object | if the print was paused, resumed or corrected | The repair seam (below). Needs `beam`. |
| `reprint` | `{time_h, filament_g}` | no | Slicer estimate for a reprint; adds its cost. |

**Feature item:** `name`; `role` = `cosmetic`, `fit` or `load_bearing`; the measurement as `readings_mm` (list)
or `measured_mm`; and exactly one limit:
- `nominal_mm` + `tolerance_mm` (symmetric; `tolerance_mm` may be left out only for a cosmetic feature, 1.0 mm),
- `range_mm`: `[lo, hi]` (converted to midpoint ± half-width for the HW5 tool),
- `min_mm` and/or `max_mm` (one-sided, checked directly).

**Beam:** `length_mm` (load line to clamp edge), `width_mm`, `nominal_thickness_mm`, and
`measured_thickness_readings_mm` (list) or `measured_thickness_mm`; `load_N` or `load_kg`; `orientation`
(`flat` or `upright`, required); `target_safety_factor` (default 2.0); `section` (`solid` default, or `skins`
with `skin_mm` = top/bottom shell thickness); `interlayer_factor` (default 0.5).

**Seam:** `z_mm` (height of the first resumed layer above the bed, below the measured thickness); `pause_min`;
optional `visible_gap`: `{length_mm, distance_from_load_mm}`; leave `distance_from_load_mm` out if unknown.

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/references/method.md
# Method behind check_part.py (also the hand fallback when the script cannot run)

Use these only if the script fails, and label the results NOT SCRIPT-VERIFIED.

**Material values** (Bambu Lab PLA/PETG Basic TDS v2.0; Stratasys ABS-M30): in-plane strength PLA 39, PETG 48,
ABS 30.8 MPa; E PLA 2.68, PETG 1.46, ABS 2.40 GPa. Across layers = in-plane × 0.5 (design choice; Ahn et al.
2002 measured 10-73 % of moulded strength across bonds).

**Features.** Range [lo, hi] -> nominal (lo+hi)/2, tolerance (hi-lo)/2. Worst reading = farthest from nominal.
Margin = tolerance - |reading - nominal|; FAIL < 0, BORDERLINE < 0.05 mm, else OK. One-sided: margin =
reading - min (or max - reading). Cosmetic out of tolerance = caution only.

**Cantilever** (fixed at the clamp edge, point load F at distance L):
M = F L; solid I = b t³/12, σ = 6 F L/(b t²); skins-only I = b (t³ - (t-2s)³)/12, σ = M (t/2)/I (lower bound);
SF = strength/σ (in-plane for flat prints, × 0.5 for upright); δ = F L³/(3 E I), caution above 2 % of L;
L/t < 4 means rough.

**Seam at height z** (flat print, carries shear): solid τ = 6 F z (t-z)/(b t³); in a sparse core
τ ≈ F/(b (t-s)); rough allowable 0.5 × 0.5 × in-plane strength. Fully debonded bound:
I_split = b (z³ + (t-z)³)/12, σ = M · max(z, t-z)/2 / I_split. Upright print: σ_seam = M |z - t/2|/I (tension
across the seam). A gap more than L/3 from the load point is in the high-moment region.

**Decision rules:** as listed in SKILL.md Step 5.

**Reprint cost** = g/1000 × $23/kg + h × $0.50/h.

In [ ]:
%%writefile skills/fdm-part-acceptance-scripts/references/example_case.json
{
  "part": "shelf hook (example only: replace every value with your own case)",
  "material": "PLA",
  "measurement_uncertainty_mm": 0.05,
  "features": [
    {
      "name": "screw hole diameter",
      "role": "fit",
      "readings_mm": [
        5.22,
        5.25
      ],
      "nominal_mm": 5.3,
      "tolerance_mm": 0.15
    },
    {
      "name": "rail slot width",
      "role": "fit",
      "readings_mm": [
        6.18
      ],
      "range_mm": [
        6.0,
        6.4
      ]
    },
    {
      "name": "back plate wall",
      "role": "load_bearing",
      "readings_mm": [
        2.9
      ],
      "min_mm": 2.4
    },
    {
      "name": "outer shell length",
      "role": "cosmetic",
      "readings_mm": [
        40.6
      ],
      "nominal_mm": 40.0
    }
  ],
  "beam": {
    "length_mm": 40,
    "width_mm": 20,
    "nominal_thickness_mm": 5.0,
    "measured_thickness_readings_mm": [
      4.55,
      4.5
    ],
    "load_kg": 2,
    "orientation": "flat",
    "target_safety_factor": 2,
    "section": "solid"
  },
  "seam": {
    "z_mm": 2.0,
    "pause_min": 3
  },
  "reprint": {
    "time_h": 2.5,
    "filament_g": 38
  }
}

**Self-test before any agent touches it.** The script checks itself against the hand-calculated HW5 demo bracket
(σ = 6 × 19.62 × 40 / (20 × 5²) = 9.42 MPa, SF = 39 / 9.42 = 4.14), the skins formula, the exact range conversion,
and that a fit feature without a tolerance is refused rather than guessed.

In [ ]:
!python3 skills/fdm-part-acceptance-scripts/scripts/check_part.py --selftest
print("\nFile sizes (lines):")
for p in sorted(pathlib.Path("skills").rglob("*")):
    if p.is_file() and "__pycache__" not in p.parts:   # the self-test leaves compiled .pyc files
        print(f"  {sum(1 for _ in open(p, encoding='utf-8')):5d}  {p}")

## 4. The task: scenario files and the prompt

`case/bracket.yaml` is our team's bracket spec, copied unchanged from the Project 2 repository.
`case/print_report.md` is the constructed scenario.

In [ ]:
%%writefile case/bracket.yaml
name: bracket
height: 8
layer_h: 0.25
geometry:
  - add: {rect: {size: [60, 30]}}
  - cut: {circle: {center: [-20, 0], r: 2.5}}
  - cut: {circle: {center: [20, 0],  r: 2.5}}
min_wall_mm: 3.0
material_volume_mm3: [11500, 16500]
holes:
  - {center: [-20, 0], diameter: [4.0, 6.5], tol_center: 1.0}
  - {center: [20, 0],  diameter: [4.0, 6.5], tol_center: 1.0}
description: A flat mounting plate bolted at its two holes; one end carries a load.
material: PLA
loads:
  - name: 100 N out-of-plane at one bolt, other bolt fixed
    supports: [{box: [17, 23, -4, 4, 0, 8]}]
    loads: [{box: [-23, -17, -4, 4, 0, 8], force_N: 100, direction: [0, 0, 1]}]
    require: {safety_factor: 1.5}

In [ ]:
%%writefile case/print_report.md
# Print report: bracket test part after an in-print correction

*Scenario for 24-679 HW6. The part spec (`bracket.yaml`) is the Project 2 team's real bracket spec, copied
unchanged from our repository. The print event and the measurements below are a constructed scenario of the
kind our recovery loop is meant to produce; they are not readings from a real print.*

## Part and use
- Part: `bracket` (see `bracket.yaml`), a 60 x 30 x 8 mm PLA plate with two 5 mm bolt holes 40 mm apart
  (centres at x = -20 and x = +20 mm).
- Use: the right bolt (x = +20) is clamped; the clamp covers x = 17 to 23 mm. The left bolt (x = -20) carries
  a load of 100 N pushing out of the plate's plane (along Z). The spec requires a safety factor of 1.5.

## Print settings (Bambu Lab P1S, Bambu Studio)
- Bambu PLA Basic, 0.25 mm layers (32 layers), printed flat: the 60 x 30 face on the bed.
- 3 wall loops (0.42 mm lines), 4 top and 4 bottom shell layers (1.0 mm each), 15 % gyroid sparse infill.

## What happened during the print
- Layer 17 (z = 4.00 to 4.25 mm): the camera check flagged under-extrusion along the left (load) end.
- The agent paused the printer after layer 17. The part stayed on the 55 °C bed for about 6 minutes while a
  person reviewed the proposal, then approved "correct": layers 18 to 32 were re-planned to return to the
  original outline, and printing resumed at z = 4.25 mm.

## Measurements of the finished part (digital calipers, 0.01 mm resolution)
| Feature | Readings (mm) |
|---|---|
| Plate thickness near the clamp (x about 17) | 7.62, 7.58 |
| Plate thickness near the load end (x about -25) | 7.70, 7.71 |
| Plate width | 30.04 |
| Left (load) hole diameter, two directions | 4.86, 5.02 |
| Right (clamped) hole diameter, two directions | 5.04, 5.08 |
| Hole centre-to-centre spacing | 40.25 |
| Material between the left hole edge and the left end of the plate | 7.00 |

## Visual inspection
- A faint seam line is visible at about z = 4.2 mm on all four side faces.
- On the front face near the load end, a fingernail catches a gap along the seam about 6 mm long.
  Elsewhere the seam feels closed.

## Reprint estimate (Bambu Studio, same settings)
- 48 min, 13.2 g of PLA.

**Run harness.** For each skill: a new workspace containing only `case/`, that one skill under
`.opencode/skills/`, and an `opencode.json` with the permissions; a new, empty OpenCode config and data directory
(so a fresh session with no memory of the other run); then `opencode run --format json` with the same model and
the same prompt. Web access and sub-agents are denied in both runs, so both answers rest on the skill and the case files. The
JSON event stream is saved and printed below as a trace: every skill invocation, file read, shell command with
its output, and file write.

In [ ]:
MODEL = "anthropic/claude-sonnet-5-5"
PROMPT_TEMPLATE = (
    "Use the {skill} skill. Our Project 2 recovery loop corrected a bracket in the middle of a print. "
    "The part spec is in case/bracket.yaml and the print report with the measurements is in case/print_report.md. "
    "Decide whether we can use this part: KEEP, TEST_BEFORE_USE or REPRINT. Show the numbers behind the decision, "
    "the assumptions you made, and what a person must check before relying on it. Save your final answer to answer.md.")
RUNS = pathlib.Path("/content/runs")


def permissions(allow_shell):
    rules = [{"action": "*", "resource": "*", "effect": "allow"},
             {"action": "webfetch", "resource": "*", "effect": "deny"},
             {"action": "websearch", "resource": "*", "effect": "deny"},
             {"action": "subagent", "resource": "*", "effect": "deny"}]
    if not allow_shell:
        rules.append({"action": "shell", "resource": "*", "effect": "deny"})
    return {"$schema": "https://opencode.ai/config.json", "permissions": rules}


def run_agent(skill, allow_shell, timeout_s=1800):
    ws = RUNS / skill
    shutil.rmtree(ws, ignore_errors=True)
    shutil.copytree(ROOT / "skills" / skill, ws / ".opencode" / "skills" / skill,
                    ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copytree(ROOT / "case", ws / "case")
    (ws / "opencode.json").write_text(json.dumps(permissions(allow_shell), indent=2))
    subprocess.run(["git", "init", "-q"], cwd=ws)
    env = os.environ.copy()
    env["PWD"] = str(ws)              # OpenCode takes the session directory from PWD, not only from cwd
    xdg = RUNS / f".opencode-home-{skill}"                  # fresh config/data/state = fresh session
    shutil.rmtree(xdg, ignore_errors=True)
    for k in ("CONFIG", "DATA", "STATE", "CACHE"):
        env[f"XDG_{k}_HOME"] = str(xdg / k.lower())
    prompt = PROMPT_TEMPLATE.format(skill=skill)
    cmd = [OPENCODE, "run", "--standalone", "--format", "json", "--auto", "--title", f"hw6-{skill}", "-m", MODEL, prompt]
    print("workspace:", ws)
    print("skills visible:", [p.name for p in (ws / ".opencode" / "skills").iterdir()])
    print("permissions:", json.dumps(permissions(allow_shell)["permissions"]))
    print("command: opencode run --standalone --format json --auto -m", MODEL, "<prompt>")
    print("prompt:", prompt, "\n")
    t0 = time.time()
    r = subprocess.run(cmd, cwd=ws, env=env, capture_output=True, text=True, timeout=timeout_s)
    (ws / "events.jsonl").write_text(r.stdout)
    (ws / "stderr.log").write_text(r.stderr)
    print(f"exit code {r.returncode}, {time.time() - t0:.0f} s, {len(r.stdout.splitlines())} events")
    if r.returncode != 0:
        print(r.stderr[-3000:])
    print("answer.md written in the workspace:", (ws / "answer.md").exists())
    return ws


def _clip(s, n):
    s = str(s or "")
    return s if len(s) <= n else s[:n] + f"\n   ... [{len(s) - n} more characters]"


def show_trace(ws, out_chars=6000):
    """Print the agent's steps from the OpenCode JSON events and total the tokens and cost."""
    tok = {"input": 0, "output": 0, "reasoning": 0, "cache_read": 0}
    cost, steps, counts = 0.0, 0, {}
    for line in (ws / "events.jsonl").read_text().splitlines():
        try:
            e = json.loads(line)
        except json.JSONDecodeError:
            continue
        t, p = e.get("type"), e.get("part", {})
        if t == "text" and p.get("text"):
            print("[agent says]", _clip(p["text"], 1500).replace("\n", "\n             "))
        elif t == "tool_use":
            tool, st = p.get("tool"), p.get("state", {})
            inp, out = st.get("input", {}), st.get("output")
            counts[tool] = counts.get(tool, 0) + 1
            if tool == "skill":
                print(f"[SKILL INVOKED] skill(id={inp.get('id')!r}) -> {len(str(out))} characters of instructions loaded")
            elif tool == "shell":
                print(f"[SHELL] $ {inp.get('command')}")
                print("   " + _clip(out, out_chars).replace("\n", "\n   "))
            elif tool in ("write", "edit"):
                print(f"[{tool.upper()}] {inp.get('path')}")
                if str(inp.get("path", "")).endswith(".json"):
                    print("   " + _clip(inp.get("content"), out_chars).replace("\n", "\n   "))
            elif tool == "read":
                print(f"[READ] {inp.get('path')}")
            else:
                print(f"[{tool}] {_clip(json.dumps(inp), 300)}")
            if st.get("status") == "error":
                print("   TOOL ERROR:", _clip(st.get("error"), 800))
        elif t == "step_finish":
            steps += 1
            cost += p.get("cost") or 0
            tk = p.get("tokens") or {}
            for k in ("input", "output", "reasoning"):
                tok[k] += tk.get(k) or 0
            tok["cache_read"] += (tk.get("cache") or {}).get("read") or 0
        elif t == "error":
            print("[ERROR]", _clip(json.dumps(e), 1500))
    summary = {"steps": steps, "tool_calls": counts, "tokens": tok, "cost_usd": round(cost, 4)}
    print("\nSUMMARY:", json.dumps(summary))
    return summary


def decision_of(text):
    m = re.search(r"\b(KEEP|TEST_BEFORE_USE|REPRINT)\b", text or "")
    return m.group(1) if m else None

## 5. Run A: Markdown-only skill (fresh session)

In [ ]:
ws_md = run_agent("fdm-part-acceptance-md", allow_shell=False)
sum_md = show_trace(ws_md)

In [ ]:
from IPython.display import Markdown, display
answer_md = (ws_md / "answer.md").read_text() if (ws_md / "answer.md").exists() else "(no answer.md written)"
print("Decision found in answer.md:", decision_of(answer_md))
display(Markdown(answer_md))

**Observation.** *(to be written in my own words after the run)*

## 6. Run B: script-assisted skill (fresh session)

In [ ]:
ws_py = run_agent("fdm-part-acceptance-scripts", allow_shell=True)
sum_py = show_trace(ws_py)

In [ ]:
answer_py = (ws_py / "answer.md").read_text() if (ws_py / "answer.md").exists() else "(no answer.md written)"
print("Decision found in answer.md:", decision_of(answer_py))
for cj in sorted(p for p in ws_py.rglob("*.json") if ".opencode" not in p.parts and p.name != "opencode.json"):
    print(f"\n{cj.relative_to(ws_py)} written during the run:")
    print(cj.read_text()[:4000])
display(Markdown(answer_py))

**Observation.** *(to be written in my own words after the run)*

## 7. Independent check

Three checks that do not rely on either agent: (a) a hand calculation in plain Python written from the textbook
formulas, sharing no code with the skill; (b) a real MCP call to my HW5 server's new tool `check_deviated_part`;
(c) a search of both answers for the reference numbers.

### 7.1 Hand calculation
Inputs I read from the scenario myself: F = 100 N at the left bolt; the clamp edge is at x = 17 mm, so the arm is
L = 17 − (−20) = 37 mm; b = 30.04 mm; the thinnest reading near the clamp t = 7.58 mm (design 8.0 mm); top and
bottom shells s = 1.0 mm; seam at z = 4.25 mm; PLA in-plane strength 39 MPa (Bambu Lab PLA Basic TDS), E = 2680 MPa.

In [ ]:
F, L, b, t, t0, s, z = 100.0, 37.0, 30.04, 7.58, 8.0, 1.0, 4.25
S_PLA, E = 39.0, 2680.0
M = F * L
sig_solid = 6 * F * L / (b * t**2)
I_skin = b * (t**3 - (t - 2*s)**3) / 12
sig_skin = M * (t / 2) / I_skin
I_split = b * (z**3 + (t - z)**3) / 12
ref = {
    "moment M (N mm)": M,
    "solid stress (MPa)": sig_solid,
    "solid safety factor": S_PLA / sig_solid,
    "design-thickness safety factor": S_PLA / (6 * F * L / (b * t0**2)),
    "skins-only stress (MPa)": sig_skin,
    "skins-only safety factor": S_PLA / sig_skin,
    "seam shear in sparse core (MPa)": F / (b * (t - s)),
    "debonded-seam safety factor": S_PLA / (M * max(z, t - z) / 2 / I_split),
    "tip deflection, solid (mm)": F * L**3 / (3 * E * b * t**3 / 12),
    "tip deflection, skins (mm)": F * L**3 / (3 * E * I_skin),
    "reprint cost ($)": 13.2 / 1000 * 23 + 0.8 * 0.5,
}
for k, v in ref.items():
    print(f"  {k:<34} {v:9.3f}")
print("\nHole limits: 4.0-6.5 mm; worst readings 4.86 and 5.04 mm -> both inside. Spacing 40.25 vs 40.0 +- 1.0 -> inside.")
print("Rules: TEST (seam crosses the load path; open gap in the low-moment end; debonded SF",
      f"{ref['debonded-seam safety factor']:.2f} < 1.5). No REPRINT rule: skins SF {ref['skins-only safety factor']:.2f} >= 1.5,",
      "debonded SF > 1. Expected decision: TEST_BEFORE_USE.")

### 7.2 MCP call to the new HW5 tool `check_deviated_part`
The same case through my HW5 MCP server over stdio (the server is installed from GitHub in §2). The tool only
models two-sided tolerances and a solid arm, so I give it exactly that; its numbers should equal the
`HW5 check_deviated_part` lines in the script's output, which shows that the skill's copy of the HW5 code behaves
like the served tool.

In [ ]:
%%writefile check/mcp_check.py
"""Independent check through the real HW5 MCP server: call the new HW5 tool `check_deviated_part` over stdio
with the bracket case (two-sided features and the solid arm only, which is all that tool models) and save the
request and response to mcp_check.json."""
import asyncio
import json
import sys

from mcp import Client, StdioServerParameters

ARGS = {
    "features": [
        # range limits 4.0-6.5 mm written as midpoint +- half-width; worst (smallest) readings
        {"name": "left (load) hole diameter", "role": "fit", "nominal_mm": 5.25, "measured_mm": 4.86, "tolerance_mm": 1.25},
        {"name": "right (clamped) hole diameter", "role": "fit", "nominal_mm": 5.25, "measured_mm": 5.04, "tolerance_mm": 1.25},
        {"name": "hole spacing", "role": "fit", "nominal_mm": 40.0, "measured_mm": 40.25, "tolerance_mm": 1.0},
    ],
    "beam": {"material": "PLA", "length_mm": 37, "width_mm": 30.04, "nominal_thickness_mm": 8.0,
             "measured_thickness_mm": 7.58, "load_N": 100, "orientation": "flat", "target_safety_factor": 1.5},
    "repaired_in_load_path": True,
    "reprint_time_h": 0.8,
    "reprint_filament_g": 13.2,
}


async def main(out="mcp_check.json"):
    params = StdioServerParameters(command=sys.executable, args=["-m", "fdm_mcp.server"])
    async with Client(params) as c:
        tools = [t.name for t in (await c.list_tools()).tools]
        r = await c.call_tool("check_deviated_part", ARGS)
    rec = {"tools_listed": tools, "request": {"tool": "check_deviated_part", "arguments": ARGS},
           "is_error": r.is_error, "structured": r.structured_content}
    with open(out, "w") as f:
        json.dump(rec, f, indent=2, ensure_ascii=False)
    return rec


if __name__ == "__main__":
    rec = asyncio.run(main())
    s = rec["structured"]
    print("tools listed:", ", ".join(rec["tools_listed"]))
    print("decision:", s["decision"], "| verdict:", s["verdict"])
    print(s["summary"])
    for k, v in s["results"].items():
        print(f"  {v['label']}: {v['value']} {v['unit']}")

In [ ]:
# run as a separate process: the stdio client launches the server as its own subprocess
r = subprocess.run([sys.executable, "mcp_check.py"], cwd="check", capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
rec = json.loads(pathlib.Path("check/mcp_check.json").read_text())
print("tools listed by the server:", ", ".join(rec["tools_listed"]))
print("request:", json.dumps(rec["request"]["arguments"])[:400], "...")
s = rec["structured"]
print("\nis_error:", rec["is_error"], "| decision:", s["decision"], "| verdict:", s["verdict"])
print(s["summary"])
for k, v in s["results"].items():
    print(f"  {v['label']}: {v['value']} {v['unit']}")

### 7.3 Which reference numbers appear in each answer?
An automatic search, so I still read both answers: a number counts as found if it appears with at least two
decimals (one for values of 10 or more) and rounds to the reference value.

In [ ]:
def found(text, v):
    need = 2 if abs(v) < 10 else 1
    for tok in re.findall(r"(?<![\d.])\d+\.\d+", text or ""):
        d = len(tok.split(".")[1])
        if d >= need and abs(float(tok) - v) <= 0.5 * 10**-d + 1e-9:
            return "yes"
    return "-"

keys = ["solid stress (MPa)", "solid safety factor", "design-thickness safety factor", "skins-only stress (MPa)",
        "skins-only safety factor", "seam shear in sparse core (MPa)", "debonded-seam safety factor",
        "tip deflection, skins (mm)", "reprint cost ($)"]
print(f"{'reference value':<34}{'value':>9}   {'A: md':>6}   {'B: scripts':>10}")
for k in keys:
    print(f"{k:<34}{ref[k]:9.3f}   {found(answer_md, ref[k]):>6}   {found(answer_py, ref[k]):>10}")
print(f"\n{'decision':<34}{'TEST_BEFORE_USE':>15}   A: {decision_of(answer_md)}   B: {decision_of(answer_py)}")
print("\nRun summaries:")
for name, sm in (("A: md", sum_md), ("B: scripts", sum_py)):
    print(f"  {name:<11} steps {sm['steps']:3d}  tools {sm['tool_calls']}  tokens {sm['tokens']}  cost ${sm['cost_usd']}")

**Saving the evidence.** The cell below bundles both workspaces (events, answers, `case.json`, `result.json`)
and the MCP record into one zip.

In [ ]:
shutil.copy("check/mcp_check.json", RUNS / "mcp_check.json")
archive = shutil.make_archive("/content/hw6_runs", "zip", RUNS)
print("saved", archive)

## 8. Conclusion

My comparison of the two versions (correctness, use of domain knowledge, reproducibility, verification,
omissions and failures, what the scripts changed and what still needed human judgement) is in my written
submission, in my own words.

**Finding.** *(to be written in my own words after the run)*

**Next step.** *(to be written in my own words after the run)*

---

### Collaboration, citation, and AI-usage disclosure

**Credit.** The bracket spec is our Project 2 team's (`parts/bracket.yaml`). Material values: Bambu Lab PLA/PETG Basic technical data sheets v2.0 and the Stratasys ABS-M30 data sheet; the interlayer factor follows Ahn et al. (2002), *Anisotropic material properties of fused deposition modeling ABS*, Rapid Prototyping Journal 8(4). OpenCode (anomalyco/opencode, MIT) runs the agent; the MCP Python SDK calls my HW5 server.

**Collaboration.** The skills, script, scenario and notebook are my own work. The bracket spec was written by our Project 2 team.

**AI usage.** Generative AI (Claude, Anthropic) was used to help draft both skills, the supporting script, the scenario files and the code in this notebook. I then read, executed, corrected and edited all of it, chose the task and the decision rules, and checked the numbers against the hand calculation and the MCP call in §7. The two agent answers in §5 and §6 are generated by Claude Sonnet 5.5 inside OpenCode and are shown unedited. My comparison of the two versions and my Project 2 brief are written by me without generative AI, as the assignment requires.

**License.** Code released under the MIT licence. Material values belong to the manufacturers' data sheets cited in the skills.